# LoanGuard AI - MLOps Lab
## Data & Model Versioning + Registry

---

### Your Mission

You've just joined **LoanGuard AI**, a fintech startup that predicts loan defaults.
Their ML team has a serious problem: models keep breaking in production and nobody can explain why.
Predictions changed last week, accuracy dropped, and a roll back took 3 days.

**Your job:** Fix their MLOps stack - layer by layer - using data versioning, experiment tracking, model versioning, and a model registry.

---

### Lab Structure

| Task | Topic | Real-World Problem You Solve |
|------|-------|------------------------------|
| **Task 1** | Data Versioning with DVC | Reproduce a model that broke 2 months ago |
| **Task 2** | Experiment Tracking with MLflow | Compare 3 model candidates before deploying |
| **Task 3** | Model Versioning | Name and register a model so the team can roll back |
| **Task 4** | Registry Lifecycle | Promote a model through staging → production → archived |

---

### How the Lab Works

- Each task has a ** Beginner path** (fill in the blanks) and a ** Advanced path** (write from scratch)
- Use the ** Hint system** - try first, then reveal hints one at a time
- Run the ** Auto-check** cells to verify your work before moving on
- **You do not need to run DVC or MLflow CLI** - everything is done in Python

---

### Setup - Run this first

## Environment Setup for LoanGuard AI Lab

**My Learnings**

The below cell initializes the working environment required for the LoanGuard AI project.
It prepares the necessary libraries, directory structure, and experiment tracking
configuration before starting the machine learning workflow.

The following operations are performed during execution:

1. Required Python libraries are imported for:
   - Data manipulation (NumPy, Pandas)
   - Machine learning models (Scikit-Learn)
   - Model tracking and experiment management (MLflow)
   - File and system operations

2. A project workspace named **loanGuard_lab** is initialized.

3. Inside the workspace, directories are created to organize project resources:
   - **data/** → stores datasets used in the lab
   - **models/** → stores trained machine learning models
   - **dvc_cache/** → used for caching artifacts or intermediate outputs

4. MLflow tracking is configured using a SQLite database so that future
experiments can log parameters, metrics, and trained models.

This setup ensures that the notebook environment is structured and ready
for the subsequent steps of the machine learning pipeline.

In [1]:
# --- SETUP - Run this cell before anything else ----------------------
import os, json, hashlib, shutil, warnings 
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import pickle, time

warnings.filterwarnings("ignore")

# Lab workspace
LAB_DIR = Path("loanGuard_lab")
LAB_DIR.mkdir(exist_ok=True)
(LAB_DIR / "data").mkdir(exist_ok=True)
(LAB_DIR / "models").mkdir(exist_ok=True)
(LAB_DIR / "dvc_cache").mkdir(exist_ok=True)
MLFLOW_URI = f"sqlite:///{LAB_DIR}/mlflow.db"

print(" Setup complete - LoanGuard AI Lab is ready")
print(f"   Workspace : {LAB_DIR.resolve()}")
print(f"   MLflow DB : {MLFLOW_URI}")

 Setup complete - LoanGuard AI Lab is ready
   Workspace : /home/ciuser/jupyterteam3/students/Swetha_Jupyter/Data Model/loanGuard_lab
   MLflow DB : sqlite:///loanGuard_lab/mlflow.db


## Observation

After running the setup cell, the project workspace was successfully initialized.

The output confirms that the **loanGuard_lab** directory was created along with
its subfolders (**data**, **models**, and **dvc_cache**). These folders help keep
datasets, trained models, and cached artifacts organized during experimentation.

In addition, an **MLflow SQLite database** was configured for experiment tracking,
allowing future model runs to record parameters, metrics, and artifacts.

This indicates that the environment is properly prepared for the upcoming
data preprocessing and model training steps.

### Simulated Data Generator

LoanGuard AI receives loan applications with customer financial data.
Run the cell below to simulate **three versions** of their dataset - representing how data changed over time.

>  **Don't modify this cell.** It simulates real-world upstream data changes you'll need to detect and version.

## Simulated Loan Application Data

**My Learnings**

The below cell generates synthetic loan application datasets used for the LoanGuard AI lab.

A function (`make_loan_dataset`) is used to simulate realistic financial data
such as income, credit score, loan amount, employment history, and debt ratio.
These variables influence the probability of loan default, which is also
generated within the dataset.

Three versions of the dataset are produced to mimic real-world data evolution:

- **v1** – baseline dataset with the original schema
- **v2** – introduces a new column (`num_accounts`) representing schema changes
- **v3** – introduces missing values in `credit_score` to simulate data quality issues

These versions help demonstrate how machine learning pipelines must handle
changing data structures and quality over time.

In [2]:
# ----- DATA SIMULATOR - Do not modify ----------------------------------
np.random.seed(42)

def make_loan_dataset(n=2000, schema_version="v1", seed=42):
    """Simulate LoanGuard's loan application data at different points in time."""
    rng = np.random.RandomState(seed)
    age            = rng.randint(22, 65, n)
    income         = rng.normal(55000, 20000, n).clip(15000, 200000)
    loan_amount    = rng.normal(12000, 5000, n).clip(1000, 50000)
    credit_score   = rng.normal(680, 80, n).clip(300, 850)
    employment_yrs = rng.randint(0, 30, n)
    debt_ratio     = rng.uniform(0.05, 0.65, n)

    # Default probability driven by risk factors
    risk = (
        -0.3 * (credit_score - 680) / 80
        + 0.4 * debt_ratio
        + 0.2 * (loan_amount / income)
        - 0.1 * employment_yrs / 10
    )
    default = (1 / (1 + np.exp(-risk * 3 + rng.normal(0, 0.5, n))) > 0.5).astype(int)

    df = pd.DataFrame({
        "age": age, "annual_income": income.round(2),
        "loan_amount": loan_amount.round(2), "credit_score": credit_score.round(1),
        "employment_years": employment_yrs, "debt_to_income": debt_ratio.round(4),
        "default": default
    })

    # v2: upstream adds a new column (common real-world schema drift)
    if schema_version in ("v2", "v3"):
        df["num_accounts"] = rng.randint(1, 10, n)

    # v3: a data quality issue introduces nulls (another common real-world problem)
    if schema_version == "v3":
        null_idx = rng.choice(n, size=int(n * 0.15), replace=False)
        df.loc[null_idx, "credit_score"] = np.nan

    return df

# Generate and save three dataset versions
datasets = {}
for ver, schema, seed in [("v1", "v1", 42), ("v2", "v2", 42), ("v3", "v3", 99)]:
    df = make_loan_dataset(schema_version=schema, seed=seed)
    path = LAB_DIR / "data" / f"loans_{ver}.csv"
    df.to_csv(path, index=False)
    datasets[ver] = df
    print(f"  loans_{ver}.csv  |  {len(df):,} rows  |  {df.shape[1]} cols  |  "
          f"{df['default'].mean():.1%} default rate  |  nulls: {df.isnull().sum().sum()}")

print("\n Three dataset versions are ready")

  loans_v1.csv  |  2,000 rows  |  7 cols  |  58.1% default rate  |  nulls: 0
  loans_v2.csv  |  2,000 rows  |  8 cols  |  58.1% default rate  |  nulls: 0
  loans_v3.csv  |  2,000 rows  |  8 cols  |  54.6% default rate  |  nulls: 300

 Three dataset versions are ready


In [3]:
# -------- Dataset Version Summary Table --------

# List to store summary information for each dataset version
summary = []

# Looping through each dataset version stored in the 'datasets' dictionary
for name, df in datasets.items():

    # Creating a dictionary capturing key statistics of the dataset
    summary.append({
        "Dataset Version": name,                     # Version label (v1, v2, v3)
        "Rows": df.shape[0],                         # Total number of rows in dataset
        "Columns": df.shape[1],                      # Total number of columns
        "Default Rate (%)": round(df["default"].mean() * 100, 2),  # Percentage of defaults
        "Missing Values": df.isnull().sum().sum()    # Total count of missing values
    })

# Converting the list of dictionaries into a pandas DataFrame
summary_df = pd.DataFrame(summary)

# Adding a serial number column for better presentation
summary_df.insert(0, "S.No", range(1, len(summary_df) + 1))

# Displaying the summary table while hiding the default pandas index
display(summary_df.style.hide(axis="index"))

S.No,Dataset Version,Rows,Columns,Default Rate (%),Missing Values
1,v1,2000,7,58.050000,0
2,v2,2000,8,58.050000,0
3,v3,2000,8,54.600000,300


### Observation

All three dataset versions contain **2000 rows**, indicating that the number of records remains consistent across versions. The **number of columns increases from 7 in v1 to 8 in v2 and v3**, showing that a new feature was introduced in later versions. While **v1 and v2 contain no missing values**, **v3 introduces 300 missing values**, representing a simulated data quality issue. The **default rate remains similar across the datasets**, with only a slight variation in v3.

In [4]:
# Inspection for each dataset 

for name, df in datasets.items():

    print("\n" + "="*50)
    print(f"{name.upper()} DATASET SCHEMA")
    print("="*50)

    schema_table = pd.DataFrame({
        "Column Name": df.columns,
        "Data Type": df.dtypes.values,
        "Non-Null Count": df.notnull().sum().values,
        "Missing Values": df.isnull().sum().values
    })

    # Insertig serial number column
    schema_table.insert(0, "S.No", range(1, len(schema_table) + 1))

    # Hiding the default pandas index for clean output
    display(schema_table.style.hide(axis="index"))


V1 DATASET SCHEMA


S.No,Column Name,Data Type,Non-Null Count,Missing Values
1,age,int64,2000,0
2,annual_income,float64,2000,0
3,loan_amount,float64,2000,0
4,credit_score,float64,2000,0
5,employment_years,int64,2000,0
6,debt_to_income,float64,2000,0
7,default,int64,2000,0



V2 DATASET SCHEMA


S.No,Column Name,Data Type,Non-Null Count,Missing Values
1,age,int64,2000,0
2,annual_income,float64,2000,0
3,loan_amount,float64,2000,0
4,credit_score,float64,2000,0
5,employment_years,int64,2000,0
6,debt_to_income,float64,2000,0
7,default,int64,2000,0
8,num_accounts,int64,2000,0



V3 DATASET SCHEMA


S.No,Column Name,Data Type,Non-Null Count,Missing Values
1,age,int64,2000,0
2,annual_income,float64,2000,0
3,loan_amount,float64,2000,0
4,credit_score,float64,1700,300
5,employment_years,int64,2000,0
6,debt_to_income,float64,2000,0
7,default,int64,2000,0
8,num_accounts,int64,2000,0


## Observation

The schema inspection confirms the structural differences between dataset versions.

Version **v1** contains 7 columns with no missing values. In **v2**, a new feature
`num_accounts` is introduced, increasing the total columns to 8. Version **v3**
retains the same schema as v2 but shows **300 missing values in the `credit_score`
column**, indicating a data quality issue.

This demonstrates both **schema evolution** and **missing data**, which are common
challenges in real-world machine learning pipelines.

---
# TASK 1 - Data Versioning with DVC
> Can you prove which data trained the broken model?
---

##  Task 1 - Data Versioning with DVC

### Scenario
> LoanGuard's model accuracy dropped from 87% to 71% last month. The engineering team says 'the code didn't change'. You suspect a silent data change. You need to (a) version the dataset snapshots so they can be reproduced later, (b) detect what changed between versions, and (c) implement schema validation to catch future changes automatically.

### Objective
By the end of this task you will be able to:
- Simulate DVC-style versioning using checksums and metadata (`.dvc` pointer files)
- Compare two dataset versions and identify schema/distribution drift
- Write a schema validator that raises an alert before a bad dataset enters the pipeline

---

###  Background - How DVC Versioning Works

DVC doesn't store data in Git. It stores a **tiny pointer file** (`.dvc`) containing:
- A **checksum** (MD5 hash) of the data file
- The file path and size
- The remote storage location

When you run `dvc pull`, DVC uses the checksum to fetch the exact data file.

In this task you'll **simulate this mechanism in pure Python** - the same logic DVC uses internally.

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  1A - Create a DVC-style pointer file

Create a function that takes a CSV file path and produces a `.dvc` metadata dictionary
containing: `md5`, `size`, `path`, `rows`, `cols`, and `created_at`.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `hashlib.md5()` to compute the file checksum. Open the file in binary mode (`'rb'`) and feed it to `.update()`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

For metadata: `os.path.getsize(path)` gives size in bytes. Load with `pd.read_csv()` and use `.shape` for rows/cols.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def create_dvc_pointer(csv_path):
    path = Path(csv_path)
    with open(path, 'rb') as f:
        md5 = hashlib.md5(f.read()).hexdigest()
    df = pd.read_csv(path)
    return {
        'md5': md5, 'path': str(path),
        'size': os.path.getsize(path),
        'rows': len(df), 'cols': df.shape[1],
        'created_at': datetime.now().isoformat()
    }
```

</details>

## Explanation

In this step, a function `create_dvc_pointer()` is implemented to simulate how
DVC tracks dataset versions. Instead of storing the dataset directly, a small
metadata pointer is created containing important information about the file.

The function computes an **MD5 checksum** of the dataset to uniquely identify
the file version. It also extracts metadata such as the file path, file size,
number of rows, and number of columns.

This metadata acts as a lightweight reference to the dataset, allowing the
exact version of data used in a machine learning pipeline to be verified and
reproduced later.

In [5]:
#  BEGINNER 1A - Create DVC-style pointer files
# ----------------------------------------------------------------------------------------

def create_dvc_pointer(csv_path):
    """
    Create a DVC-style metadata pointer for a dataset file.

    Parameters
    ----------
    csv_path : str or Path
        Path to the CSV file to version.

    Returns
    -------
    dict  with keys: md5, path, size, rows, cols, created_at
    """
    path = Path(csv_path)

    # Step 1: Compute MD5 checksum of the file
    with open(path, "rb") as f:                 # open file in binary mode
        md5 = hashlib.md5(f.read()).hexdigest() # compute file checksum

    # Step 2: Load the CSV to get row/column counts
    df = pd.read_csv(path)                      # read dataset into DataFrame

    # Step 3: Build and return the metadata dictionary
    return {
        "md5":        md5,
        "path":       str(path),
        "size":       os.path.getsize(path),    # file size in bytes
        "rows":       len(df),                  # total rows
        "cols":       df.shape[1],              # total columns
        "created_at": datetime.now().isoformat()
    }

# ----- Test it -----
pointer_v1 = create_dvc_pointer(LAB_DIR / "data" / "loans_v1.csv")

print("\n" + "="*50)
print("DVC POINTER METADATA (loans_v1)")
print("="*50)

print(json.dumps(pointer_v1, indent=2))


DVC POINTER METADATA (loans_v1)
{
  "md5": "32749950b434fb1f5e15bc9573a4ccd0",
  "path": "loanGuard_lab/data/loans_v1.csv",
  "size": 76059,
  "rows": 2000,
  "cols": 7,
  "created_at": "2026-03-13T14:20:35.998503"
}


## Observation

The generated DVC-style pointer captures key metadata for the dataset
`loans_v1.csv`. It records the file checksum (MD5), file path, size,
row count (2000), and column count (7), along with a timestamp.

This metadata acts as a lightweight reference to the dataset version,
allowing the exact data used for model training to be verified or
reproduced later.

###  1B - Version all three datasets and save pointer files

Call your function on all three loan datasets and save each pointer as a `.dvc` JSON file.

## Explanation

In this step, the previously created `create_dvc_pointer()` function is used to
generate version metadata for all three dataset snapshots (`v1`, `v2`, and `v3`).

Each dataset is processed to create a **DVC-style pointer**, which stores
metadata such as the dataset checksum (MD5), file size, number of rows,
and number of columns. The checksum acts as a unique fingerprint that allows
a dataset version to be verified later.

After generating the pointer, it is saved as a `.dvc` JSON file. These pointer
files simulate how **DVC tracks dataset versions without storing the actual
data inside Git**, enabling reproducibility of machine learning experiments.

In [6]:
#  BEGINNER 1B - Version all three datasets
# --------------------------------------------------

dvc_store = {}   # will hold {version: pointer_dict}

for version in ["v1", "v2", "v3"]:
    csv_path = LAB_DIR / "data" / f"loans_{version}.csv"

    # Step 1: Create pointer using your function above
    pointer = create_dvc_pointer(csv_path)

    # Step 2: Save pointer to a .dvc JSON file
    dvc_file = LAB_DIR / "data" / f"loans_{version}.csv.dvc"
    with open(dvc_file, "w") as f:
        json.dump(pointer, f, indent=2)

    dvc_store[version] = pointer
    print(f"  loans_{version}.csv  →  md5: {pointer['md5'][:12]}...  "
          f"({pointer['rows']} rows, {pointer['cols']} cols)")

print("\n All versions are tracked")

  loans_v1.csv  →  md5: 32749950b434...  (2000 rows, 7 cols)
  loans_v2.csv  →  md5: 3b43198058ae...  (2000 rows, 8 cols)
  loans_v3.csv  →  md5: b8ac0fe8ebdd...  (2000 rows, 8 cols)

 All versions are tracked


## Observation

The function was successfully applied to all three dataset versions (v1, v2, and v3).
For each dataset, a DVC-style pointer file was generated and saved as a `.dvc`
JSON file. The output confirms that each dataset now has a unique **MD5 checksum**
along with its row and column metadata.

These pointer files allow the exact dataset version used in a machine learning
pipeline to be identified and reproduced later.

###  1C - Compare two versions and detect drift

Write a function that compares two DVC pointers and the underlying CSVs to detect:
- Schema changes (new/removed columns)
- Row count changes
- Statistical drift in numeric columns (mean shift > 5%)
- Data quality issues (null counts)

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Load both CSVs with `pd.read_csv()`. Compare `df1.columns` and `df2.columns` using set operations to find added/removed columns.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

For numeric drift, iterate over shared columns: `abs(df2[col].mean() - df1[col].mean()) / (df1[col].mean() + 1e-9)`. Flag anything above 0.05.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def compare_versions(ptr_a, ptr_b):
    df_a, df_b = pd.read_csv(ptr_a['path']), pd.read_csv(ptr_b['path'])
    cols_a, cols_b = set(df_a.columns), set(df_b.columns)
    report = {
        'added_cols':   list(cols_b - cols_a),
        'removed_cols': list(cols_a - cols_b),
        'row_delta':    ptr_b['rows'] - ptr_a['rows'],
        'null_delta':   int(df_b.isnull().sum().sum() - df_a.isnull().sum().sum()),
        'drift': {}
    }
    for col in cols_a & cols_b:
        if df_a[col].dtype in [float, int]:
            shift = abs(df_b[col].mean() - df_a[col].mean()) / (abs(df_a[col].mean()) + 1e-9)
            if shift > 0.05:
                report['drift'][col] = round(shift, 4)
    return report
```

</details>

## Explanation

In this step, a function is implemented to compare two versioned datasets and
detect potential data drift. The comparison uses both the dataset metadata
stored in the DVC-style pointers and the actual CSV files.

The function checks for several types of changes between dataset versions:
- **Schema changes** by identifying added or removed columns.
- **Row count changes** to detect differences in dataset size.
- **Data quality issues** by comparing total missing values.
- **Statistical drift** in numeric columns by measuring relative shifts in
  column means. A column is flagged if the mean changes by more than **5%**.

These checks help identify silent changes in the data that could affect
machine learning model performance.

In [7]:
#  BEGINNER 1C - Detect drift between dataset versions
# ----------------------------------------------------------

def compare_dataset_versions(ptr_a, ptr_b):
    """
    Compare two versioned datasets and produce a drift report.

    Returns a dict with:
      - added_cols    : columns present in b but not a
      - removed_cols  : columns present in a but not b
      - row_delta     : change in row count
      - null_delta    : change in total null count
      - drift         : dict of {col: relative_mean_shift} for drifted columns
    """
    df_a = pd.read_csv(ptr_a["path"])
    df_b = pd.read_csv(ptr_b["path"])

    cols_a = set(df_a.columns)
    cols_b = set(df_b.columns)

    report = {
        "added_cols":   list(cols_b - cols_a),
        "removed_cols": list(cols_a - cols_b),
        "row_delta":    ptr_b["rows"] - ptr_a["rows"],
        "null_delta":   int(df_b.isnull().sum().sum() - df_a.isnull().sum().sum()),
        "drift":        {}
    }

    # Detect numeric drift in shared columns
    shared_numeric = [
        c for c in cols_a & cols_b
        if df_a[c].dtype in [np.float64, np.int64]
    ]
    for col in shared_numeric:
        mean_a = df_a[col].mean()
        mean_b = df_b[col].mean()
        # YOUR CODE HERE - compute relative shift and add to report if > 5%
        shift = abs(mean_b - mean_a) / (abs(mean_a) + 1e-9)
        if shift > 0.05:
            report["drift"][col] = round(shift, 4)

    return report

# ----- Compare v1 → v2, then v2 → v3 -----
print("═" * 55)
print("Comparing loans_v1  →  loans_v2")
print("═" * 55)
r12 = compare_dataset_versions(dvc_store["v1"], dvc_store["v2"])
print(json.dumps(r12, indent=2))

print("\n" + "═" * 55)
print("Comparing loans_v2  →  loans_v3  (the bad version!)")
print("═" * 55)
r23 = compare_dataset_versions(dvc_store["v2"], dvc_store["v3"])
print(json.dumps(r23, indent=2))

═══════════════════════════════════════════════════════
Comparing loans_v1  →  loans_v2
═══════════════════════════════════════════════════════
{
  "added_cols": [
    "num_accounts"
  ],
  "removed_cols": [],
  "row_delta": 0,
  "null_delta": 0,
  "drift": {}
}

═══════════════════════════════════════════════════════
Comparing loans_v2  →  loans_v3  (the bad version!)
═══════════════════════════════════════════════════════
{
  "added_cols": [],
  "removed_cols": [],
  "row_delta": 0,
  "null_delta": 300,
  "drift": {
    "default": 0.0594
  }
}


## Observation

The comparison between **v1 and v2** shows a schema change where the column
`num_accounts` was added, while the row count and null values remained unchanged.

The comparison between **v2 and v3** shows no schema changes, but a **data quality
issue** where the total number of null values increased by **300**. Additionally,
a statistical drift is detected in the **`default` column** with a mean shift of
approximately **5.94%**, indicating a change in the distribution of the target
variable.

These results demonstrate how version comparison can identify **schema changes,
data quality problems, and statistical drift** between dataset versions.

###  1D - Schema Validator

Write a schema validator that checks incoming data against a **golden schema** (captured from a known-good dataset).
It must raise a `ValueError` with a descriptive message if validation fails.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Store the golden schema as a dict: `{'columns': list, 'dtypes': dict, 'null_thresholds': dict}`. Compare incoming data against it.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Check three things in order: (1) all expected columns present, (2) null rate per column below threshold, (3) value ranges reasonable.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def build_golden_schema(df, null_tolerance=0.05):
    return {
        'columns': list(df.columns),
        'null_rates': {c: df[c].isnull().mean() for c in df.columns},
        'null_tolerance': null_tolerance,
        'ranges': {c: (df[c].min(), df[c].max())
                   for c in df.select_dtypes(include=np.number).columns}
    }

def validate_schema(df, schema):
    errors = []
    missing = set(schema['columns']) - set(df.columns)
    if missing: errors.append(f'Missing columns: {missing}')
    for col in schema['columns']:
        if col in df.columns:
            null_rate = df[col].isnull().mean()
            if null_rate > schema['null_tolerance']:
                errors.append(f'{col}: null rate {null_rate:.1%} > tolerance')
    if errors:
        raise ValueError('Schema validation failed:\n' + '\n'.join(errors))
    return True
```

</details>

## Explanation

In this step, a schema validation mechanism is implemented to ensure that
incoming datasets follow the expected structure of a known-good dataset,
referred to as the **golden schema**.

The function `build_golden_schema()` captures important schema information
from the reference dataset, including the list of expected columns, the
observed null rates for each column, and acceptable value ranges for numeric
features.

The function `validate_schema()` then checks a new dataset against this schema.
It verifies that all required columns are present and that the proportion of
missing values in each column does not exceed the allowed tolerance. If any
violations are detected, the function raises a `ValueError` listing all schema
issues.

This validation step helps prevent corrupted or unexpected data from entering
the machine learning pipeline.

In [8]:
#  BEGINNER 1D - Schema Validator
# ----------------------------------------------------------

def build_golden_schema(df, null_tolerance=0.05):
    """
    Capture the schema of a known-good dataset.

    Returns a dict with: columns, null_rates, null_tolerance, ranges
    """
    return {
        "columns": list(df.columns),
        "null_rates": {c: df[c].isnull().mean() for c in df.columns},
        "null_tolerance": null_tolerance,
        "ranges": {
            c: (df[c].min(), df[c].max())
            for c in df.select_dtypes(include=np.number).columns
        }
    }


def validate_schema(df, schema):
    """
    Validate a new DataFrame against a golden schema.
    Raise ValueError if validation fails, listing ALL errors found.
    """
    errors = []

    # Check 1: All expected columns are present
    missing = set(schema["columns"]) - set(df.columns)
    if missing:
        errors.append(f"Missing columns: {missing}")

    # Check 2: Null rate per column is within tolerance
    for col in schema["columns"]:
        if col in df.columns:
            null_rate = df[col].isnull().mean()
            if null_rate > schema["null_tolerance"]:
                errors.append(f"{col}: null rate {null_rate:.1%} > tolerance")

    if errors:
        raise ValueError("Schema validation FAILED:\n" + "\n".join(f"   {e}" for e in errors))

    print(" Schema validation passed")
    return True


# ----- Test: v1 is the golden schema -----

print("═" * 55)
print("Building Golden Schema (from loans_v1)")
print("═" * 55)

golden = build_golden_schema(datasets["v1"])
print("Columns:", golden.get("columns", []))
print("Null tolerance:", golden.get("null_tolerance", None))
print()

print("═" * 55)
print("Validating loans_v2 (extra column)")
print("═" * 55)

try:
    validate_schema(datasets["v2"], golden)
except ValueError as e:
    print(f"Caught validation error:\n{e}")

print()

print("═" * 55)
print("Validating loans_v3 (null issue expected)")
print("═" * 55)

try:
    validate_schema(datasets["v3"], golden)
except ValueError as e:
    print(f"Caught validation error:\n{e}")

═══════════════════════════════════════════════════════
Building Golden Schema (from loans_v1)
═══════════════════════════════════════════════════════
Columns: ['age', 'annual_income', 'loan_amount', 'credit_score', 'employment_years', 'debt_to_income', 'default']
Null tolerance: 0.05

═══════════════════════════════════════════════════════
Validating loans_v2 (extra column)
═══════════════════════════════════════════════════════
 Schema validation passed

═══════════════════════════════════════════════════════
Validating loans_v3 (null issue expected)
═══════════════════════════════════════════════════════
Caught validation error:
Schema validation FAILED:
   credit_score: null rate 15.0% > tolerance


## Observation

The golden schema was successfully built using dataset **v1**, capturing the
expected columns and acceptable null tolerance.

When validating **v2**, the dataset passed schema validation because all
required columns were present and null rates were within the allowed limit,
even though an additional column (`num_accounts`) exists.

When validating **v3**, the validation failed due to a **data quality issue**.
The `credit_score` column contained a **15.0% null rate**, which exceeds the
allowed tolerance of **5%**, triggering a schema validation error.

In [9]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert pointer_v1.get('md5') and len(pointer_v1['md5']) == 32, 'md5 must be a 32-char hex string'
    passed.append('DVC pointer created')
except Exception as e:
    failed.append(('DVC pointer created', str(e)))

try:
    assert all(k in pointer_v1 for k in ['md5','path','size','rows','cols']), 'Missing keys in pointer'
    passed.append('DVC pointer has required keys')
except Exception as e:
    failed.append(('DVC pointer has required keys', str(e)))

try:
    assert len(dvc_store) == 3, 'dvc_store should have v1, v2, v3'
    passed.append('All 3 versions stored')
except Exception as e:
    failed.append(('All 3 versions stored', str(e)))

try:
    assert r23.get('null_delta', 0) > 0, 'Should detect null increase in v3'
    passed.append('Drift detected v2→v3')
except Exception as e:
    failed.append(('Drift detected v2→v3', str(e)))

try:
    assert True, 'Manually verify the validator raised an error for v3'
    passed.append('Schema validator catches v3')
except Exception as e:
    failed.append(('Schema validator catches v3', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 5/5
  DVC pointer created
  DVC pointer has required keys
  All 3 versions stored
  Drift detected v2→v3
  Schema validator catches v3

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 1

Implement a complete DVC-style versioning system with the following requirements:

**Requirements:**
1. `create_dvc_pointer(path)` - MD5 checksum, full metadata, saved to `.dvc` file
2. `compare_dataset_versions(ptr_a, ptr_b)` - schema diff, row diff, null diff, per-column statistical drift (mean, std, min, max shifts)
3. `build_golden_schema(df)` + `validate_schema(df, schema)` - catches missing columns, null rate violations, and out-of-range values for numeric columns
4. `reproduce_dataset(version)` - given a version string, loads the correct CSV using the `.dvc` pointer file (simulating `dvc pull`)
5. Demonstrate: show that v3 fails validation, v1 passes, and `reproduce_dataset('v1')` returns the exact same checksum as the v1 pointer

Write all functions in the cell below with no scaffolding.

## Explanation

In the advanced implementation, a complete DVC-style data versioning workflow
is implemented from scratch. The system tracks dataset versions using metadata
pointer files rather than storing the datasets directly.

The `create_dvc_pointer()` function computes a checksum and stores dataset
metadata in a `.dvc` file. The `compare_dataset_versions()` function detects
schema changes, row differences, null differences, and statistical drift
between dataset versions.

A **golden schema** is built from a trusted dataset using
`build_golden_schema()`. Incoming datasets are validated using
`validate_schema()` to detect missing columns, excessive null rates, or
out-of-range values.

Finally, `reproduce_dataset()` simulates how DVC retrieves the correct dataset
version by reading the pointer file and loading the associated CSV.

In [10]:
#  ADVANCED - Task 1 Full Implementation
# ---------------------------------------
# YOUR CODE HERE
# ---------------------------------------

import json
import hashlib
import os
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime


# ------------------------------------------------
#  Create DVC Pointer
# ------------------------------------------------
def create_dvc_pointer(path):

    path = Path(path)

    with open(path, "rb") as f:
        md5 = hashlib.md5(f.read()).hexdigest()

    df = pd.read_csv(path)

    pointer = {
        "md5": md5,
        "path": str(path),
        "size": os.path.getsize(path),
        "rows": len(df),
        "cols": df.shape[1],
        "created_at": datetime.now().isoformat()
    }

    dvc_file = str(path) + ".dvc"

    with open(dvc_file, "w") as f:
        json.dump(pointer, f, indent=2)

    return pointer


# ------------------------------------------------
#  Compare Dataset Versions
# ------------------------------------------------
def compare_dataset_versions(ptr_a, ptr_b):

    df_a = pd.read_csv(ptr_a["path"])
    df_b = pd.read_csv(ptr_b["path"])

    cols_a = set(df_a.columns)
    cols_b = set(df_b.columns)

    report = {
        "added_cols": list(cols_b - cols_a),
        "removed_cols": list(cols_a - cols_b),
        "row_delta": len(df_b) - len(df_a),
        "null_delta": int(df_b.isnull().sum().sum() - df_a.isnull().sum().sum()),
        "drift": {}
    }

    shared_cols = cols_a & cols_b

    for col in shared_cols:
        if df_a[col].dtype in [np.float64, np.int64]:

            stats_a = {
                "mean": df_a[col].mean(),
                "std": df_a[col].std(),
                "min": df_a[col].min(),
                "max": df_a[col].max()
            }

            stats_b = {
                "mean": df_b[col].mean(),
                "std": df_b[col].std(),
                "min": df_b[col].min(),
                "max": df_b[col].max()
            }

            shifts = {
                k: abs(stats_b[k] - stats_a[k]) / (abs(stats_a[k]) + 1e-9)
                for k in stats_a
            }

            drift_cols = {k: round(v, 4) for k, v in shifts.items() if v > 0.05}

            if drift_cols:
                report["drift"][col] = drift_cols

    return report


# ------------------------------------------------
#  Build Golden Schema
# ------------------------------------------------
def build_golden_schema(df, null_tolerance=0.05):

    return {
        "columns": list(df.columns),
        "null_rates": {c: df[c].isnull().mean() for c in df.columns},
        "null_tolerance": null_tolerance,
        "ranges": {
            c: (df[c].min(), df[c].max())
            for c in df.select_dtypes(include=np.number).columns
        }
    }


# ------------------------------------------------
# Validate Schema
# ------------------------------------------------
def validate_schema(df, schema):

    errors = []

    missing = set(schema["columns"]) - set(df.columns)
    if missing:
        errors.append(f"Missing columns: {missing}")

    for col in schema["columns"]:
        if col in df.columns:

            null_rate = df[col].isnull().mean()

            if null_rate > schema["null_tolerance"]:
                errors.append(f"{col}: null rate {null_rate:.1%} > tolerance")

            if col in schema["ranges"]:
                min_val, max_val = schema["ranges"][col]

                if df[col].min() < min_val or df[col].max() > max_val:
                    errors.append(f"{col}: values out of expected range")

    if errors:
        raise ValueError("Schema validation FAILED:\n" + "\n".join(errors))

    return True


# ------------------------------------------------
#  Reproduce Dataset
# ------------------------------------------------
def reproduce_dataset(version):

    pointer_path = LAB_DIR / "data" / f"loans_{version}.csv.dvc"

    with open(pointer_path) as f:
        pointer = json.load(f)

    df = pd.read_csv(pointer["path"])

    with open(pointer["path"], "rb") as f:
        md5 = hashlib.md5(f.read()).hexdigest()

    return df, md5, pointer["md5"]


# ------------------------------------------------
# Demonstration
# ------------------------------------------------

ptr_v1 = create_dvc_pointer(LAB_DIR / "data" / "loans_v1.csv")
ptr_v2 = create_dvc_pointer(LAB_DIR / "data" / "loans_v2.csv")
ptr_v3 = create_dvc_pointer(LAB_DIR / "data" / "loans_v3.csv")

print("="*55)
print("Comparing v1 → v2")
print("="*55)
print(json.dumps(compare_dataset_versions(ptr_v1, ptr_v2), indent=2))

print("\n" + "="*55)
print("Comparing v2 → v3")
print("="*55)
print(json.dumps(compare_dataset_versions(ptr_v2, ptr_v3), indent=2))

golden = build_golden_schema(pd.read_csv(ptr_v1["path"]))

print("\n" + "="*55)
print("Validating v1 (expected to pass)")
print("="*55)
try:
    if validate_schema(pd.read_csv(ptr_v1["path"]), golden):
        print("Schema validation passed")
except ValueError as e:
    print(e)

print("\n" + "="*55)
print("Validating v3 (expected to fail)")
print("="*55)
try:
    validate_schema(pd.read_csv(ptr_v3["path"]), golden)
except ValueError as e:
    print(e)

print("\n" + "="*55)
print("Reproducing dataset v1")
print("="*55)

df, current_md5, expected_md5 = reproduce_dataset("v1")

print("Checksum verification:", "MATCH" if current_md5 == expected_md5 else "MISMATCH")

Comparing v1 → v2
{
  "added_cols": [
    "num_accounts"
  ],
  "removed_cols": [],
  "row_delta": 0,
  "null_delta": 0,
  "drift": {}
}

Comparing v2 → v3
{
  "added_cols": [],
  "removed_cols": [],
  "row_delta": 0,
  "null_delta": 300,
  "drift": {
    "credit_score": {
      "min": 0.1951
    },
    "loan_amount": {
      "max": 0.124
    },
    "default": {
      "mean": 0.0594
    }
  }
}

Validating v1 (expected to pass)
Schema validation passed

Validating v3 (expected to fail)
Schema validation FAILED:
loan_amount: values out of expected range
credit_score: null rate 15.0% > tolerance
credit_score: values out of expected range

Reproducing dataset v1
Checksum verification: MATCH


---
# TASK 2 - Experiment Tracking with MLflow
> Which model should go to production - and can you prove it?
---

##  Task 2 - Experiment Tracking with MLflow

### Scenario
> LoanGuard's data science lead trained three models last sprint - Logistic Regression, Random Forest, and Gradient Boosting. She can't remember which hyperparameters she used for the best one, and the comparison was done in a Slack message that got deleted. You need to run all three as tracked experiments so the decision is transparent and auditable.

### Objective
By the end of this task you will be able to:
- Create an MLflow experiment and log runs with parameters, metrics, and artifacts
- Compare multiple model runs programmatically
- Identify the best model using a business-relevant metric (F1 score for imbalanced default data)
- Log the training dataset version alongside model runs for full traceability

---

###  Background - MLflow Tracking

MLflow Tracking records every experiment run as a **first-class object** with:
- **Parameters** - hyperparameters, config values
- **Metrics** - accuracy, F1, AUC, etc.
- **Artifacts** - model files, plots, data samples
- **Tags** - free-form metadata (dataset version, author, etc.)

Each run gets a unique `run_id` that links it permanently to what produced it.

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  2A - Set up MLflow and prepare data

Configure MLflow tracking and split the versioned dataset into train/test sets.

## Explanation

In this step, MLflow tracking is configured and the dataset is prepared for
model training. The MLflow tracking URI is set to the local SQLite database so
that all experiment runs, parameters, metrics, and artifacts are stored in a
persistent tracking store.

An experiment named **"loandefault_prediction"** is created (or retrieved if it
already exists) to organize all model training runs under a single experiment.

The dataset **loans_v1.csv** is then loaded and cleaned by removing rows with
missing values. Relevant feature columns and the target variable (`default`)
are separated, and the data is split into **training and testing sets (80/20)**
using a fixed random state to ensure reproducibility of the experiment.

### Why Experiment Tracking is Important

In machine learning projects, multiple models and parameter combinations are tested.  
Experiment tracking helps record metrics, parameters, and artifacts so that models can be compared and reproduced later.

MLflow provides a centralized way to:

• track model runs  
• log evaluation metrics  
• store trained model artifacts  
• compare multiple experiments

This enables data scientists to identify the best-performing model and maintain reproducibility.

In [11]:
#  BEGINNER 2A - Configure MLflow and prepare data
# ----------------------------------------------------

# Step 1: Set the MLflow tracking URI to your local SQLite database
mlflow.set_tracking_uri(MLFLOW_URI)

# Step 2: Create (or get) an experiment called "loandefault_prediction"
experiment = mlflow.set_experiment("loandefault_prediction")
experiment_id = experiment.experiment_id

# Step 3: Load loans_v1.csv and split into train / test (80/20, random_state=42)
df = pd.read_csv(LAB_DIR / "data" / "loans_v1.csv").dropna()

FEATURES = ["age", "annual_income", "loan_amount",
            "credit_score", "employment_years", "debt_to_income"]
TARGET   = "default"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
print("="*50)
print("MLflow Experiment Setup Complete")
print("="*50)
print(f"Experiment ready")
print(f"Train size : {len(X_train) if X_train is not None else '?':,}")
print(f"Test size  : {len(X_test) if X_test is not None else '?':,}")
print(f"Default rate (train): {y_train.mean():.1%}" if y_train is not None else "")

MLflow Experiment Setup Complete
Experiment ready
Train size : 1,600
Test size  : 400
Default rate (train): 57.6%


## Observation

MLflow tracking was successfully configured using the local SQLite database.
Since the experiment **"loandefault_prediction"** did not previously exist,
MLflow automatically created it.

The dataset `loans_v1.csv` was loaded and split into **1,600 training samples**
and **400 testing samples** using an 80/20 split. The training data shows a
default rate of **57.6%**, confirming that the dataset contains a relatively
high proportion of default cases for model training.

###  2B - Log a single MLflow run

Write a function that trains one model and logs everything to MLflow: params, metrics, the model artifact, and a tag for the dataset version.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `with mlflow.start_run(run_name=...) as run:` to start a run. Inside: `mlflow.log_param(key, val)` and `mlflow.log_metric(key, val)`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Log the model with `mlflow.sklearn.log_model(model, artifact_path='model')`. Add tags with `mlflow.set_tag(key, val)`.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def run_experiment(model, model_name, params, X_tr, X_te, y_tr, y_te, dataset_version):
    with mlflow.start_run(run_name=model_name) as run:
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        proba = model.predict_proba(X_te)[:, 1]

        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy':  accuracy_score(y_te, preds),
            'precision': precision_score(y_te, preds),
            'recall':    recall_score(y_te, preds),
            'f1':        f1_score(y_te, preds),
            'auc':       roc_auc_score(y_te, proba)
        })
        mlflow.set_tag('dataset_version', dataset_version)
        mlflow.set_tag('model_type', model_name)
        mlflow.sklearn.log_model(model, 'model')
        return run.info.run_id, f1_score(y_te, preds)
```

</details>

## Explanation

In this step, a reusable experiment function is implemented to train a model
and log the entire run to MLflow.

The function trains the given model, generates predictions on the test set,
and records the experiment details. Hyperparameters are logged as **MLflow
parameters**, while evaluation results (accuracy, precision, recall, F1, and
AUC) are logged as **metrics**.

Additional metadata such as the **dataset version** and **model type** are
stored as MLflow tags for traceability. The trained model is also logged as an
artifact so it can be reproduced or deployed later.

This ensures every model training run is fully tracked and comparable.

In [12]:
#  BEGINNER 2B - Log a single MLflow run
# -----------------------------------------------------

def run_experiment(model, model_name, params, X_tr, X_te, y_tr, y_te, dataset_version):
    """
    Train a model and log a complete MLflow run.

    Must log:
      - All items in `params` dict as MLflow parameters
      - accuracy, precision, recall, f1, auc as MLflow metrics
      - The fitted model as an artifact
      - 'dataset_version' and 'model_type' as tags

    Returns
    -------
    (run_id : str, f1 : float)
    """
    with mlflow.start_run(run_name=model_name) as run:

        # Step 1: Train the model
        model.fit(X_tr, y_tr)  # fit the model on the training data

        # Step 2: Generate predictions and probabilities
        preds = model.predict(X_te)                 # predicted class labels
        proba = model.predict_proba(X_te)[:, 1]     # probability of positive class (needed for AUC)

        # Step 3: Log parameters
        mlflow.log_params(params)  # record hyperparameters used for this run

        # Step 4: Log metrics - accuracy, precision, recall, f1, auc
        acc  = accuracy_score(y_te, preds)          # overall prediction accuracy
        prec = precision_score(y_te, preds)         # precision for positive class
        rec  = recall_score(y_te, preds)            # recall for positive class
        f1   = f1_score(y_te, preds)                # F1 score (important for imbalanced data)
        auc  = roc_auc_score(y_te, proba)           # ROC-AUC using predicted probabilities

        # log each evaluation metric to MLflow
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("auc", auc)

        # Step 5: Set tags for dataset_version and model_type
        mlflow.set_tag("dataset_version", dataset_version)  # track which dataset version trained the model
        mlflow.set_tag("model_type", model_name)             # record model name/type for experiment tracking

        # Step 6: Log the model artifact
        mlflow.sklearn.log_model(model, "model")  # store the trained model as an MLflow artifact

        f1 = f1_score(y_te, preds) if preds is not None else 0.0
        return run.info.run_id, f1


# ----- Test with one model -----
test_model  = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
test_params = {"C": 1.0, "max_iter": 1000, "solver": "lbfgs"}
run_id, f1  = run_experiment(
    test_model, "LogisticRegression_test", test_params,
    X_train, X_test, y_train, y_test, dataset_version="v1"
)
print("="*50)
print("MLflow Run Complete")
print("="*50)
print(f"Run ID : {run_id}")
print(f"F1     : {f1:.4f}")

2026/03/13 14:20:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 14:20:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MLflow Run Complete
Run ID : 5af3f26036184e58a048c3e9b32f5071
F1     : 0.8706


## Observation

The MLflow run was successfully executed for the Logistic Regression model.
The experiment logged the model parameters, evaluation metrics, dataset
version tag, and the trained model artifact.

The model achieved an **F1 score of 0.8706** on the test dataset, and the run
was assigned a unique **run ID**, enabling the experiment to be tracked and
reproduced through MLflow.

During execution, MLflow displayed warnings related to the use of the
`artifact_path` argument and the use of pickle-based model serialization.
These warnings are informational and do not affect the correctness of the
experiment run in this lab environment.

###  2C - Run all three candidate models

Run all three model types with the parameters below and collect their run IDs and F1 scores.

## Explanation

In this step, three candidate models are trained and tracked as separate
MLflow runs. Each model configuration is stored in the `candidates` list
along with its name and hyperparameters.

The code iterates through each candidate model and calls the previously
defined `run_experiment()` function. This function trains the model,
logs its parameters, evaluation metrics, dataset version, and model
artifact to MLflow.

The results from each run are stored in a list containing the model name,
run ID, and F1 score. These results are then displayed in a comparison
table, allowing the best-performing model to be identified based on
its F1 score.

In [13]:
#  BEGINNER 2C - Run all three candidate models
# ------------------------------------------------------

# Three candidate models with their hyperparameter configs
candidates = [
    (
        LogisticRegression(C=0.5, max_iter=1000, random_state=42),
        "LogisticRegression",
        {"model": "LogisticRegression", "C": 0.5, "max_iter": 1000}
    ),
    (
        RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        "RandomForest",
        {"model": "RandomForest", "n_estimators": 100, "max_depth": 6}
    ),
    (
        GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                   max_depth=4, random_state=42),
        "GradientBoosting",
        {"model": "GradientBoosting", "n_estimators": 100,
         "learning_rate": 0.1, "max_depth": 4}
    ),
]

# YOUR CODE HERE - loop over candidates and call run_experiment for each
# Store results as a list of dicts: [{"name": ..., "run_id": ..., "f1": ...}]
results = []

for model, name, params in candidates:

    # run the experiment and log it to MLflow
    run_id, f1 = run_experiment(
        model,
        name,
        params,
        X_train,
        X_test,
        y_train,
        y_test,
        dataset_version="v1"
    )

    # store results for later comparison
    results.append({
        "name": name,
        "run_id": run_id,
        "f1": f1
    })

# ----- Print comparison -----
print("\n" + "═"*55)
print(f"{'Model':<25} {'F1 Score':>10} {'Run ID':>15}")
print("═"*55)
for r in results:
    print(f"{r['name']:<25} {r['f1']:>10.4f}  {r['run_id'][:12]}...")
print("═"*55)

if results:
    best = max(results, key=lambda x: x["f1"])
    print(f"\nBest model: {best['name']}  (F1 = {best['f1']:.4f})")
    BEST_RUN_ID = best["run_id"]
    BEST_MODEL_NAME = best["name"]

2026/03/13 14:20:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 14:20:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 14:20:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 14:20:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p


═══════════════════════════════════════════════════════
Model                       F1 Score          Run ID
═══════════════════════════════════════════════════════
LogisticRegression            0.8706  dcc394121c9a...
RandomForest                  0.8747  94bf570a33da...
GradientBoosting              0.8722  4b8576b48f97...
═══════════════════════════════════════════════════════

Best model: RandomForest  (F1 = 0.8747)


## Observation

All three candidate models were successfully trained and logged as
separate MLflow experiment runs. Their performance was compared using
the F1 score on the test dataset.

The results show that **RandomForest achieved the highest F1 score
(0.8747)**, slightly outperforming Logistic Regression and Gradient
Boosting. Based on this metric, RandomForest is selected as the
best-performing model for the current dataset.

During execution, MLflow displayed warnings related to deprecated
arguments and model serialization methods. These warnings are
informational and do not affect the correctness of the experiment
runs in this lab environment.

###  2D - Retrieve the best model from MLflow

Use the MLflow client to load the best model back from the tracking server - as if you're another engineer picking up the work.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `MlflowClient()` to interact with the tracking server. `client.get_run(run_id)` retrieves run metadata.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

`mlflow.sklearn.load_model(f'runs:/{run_id}/model')` loads the model artifact from a run.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
client = MlflowClient()
run    = client.get_run(BEST_RUN_ID)
print(run.data.metrics)
model  = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")
preds  = model.predict(X_test)
```

</details>

## Explanation

In this step, the best-performing model is retrieved from MLflow using the
stored run ID. The MLflow tracking server stores all experiment runs along
with their parameters, metrics, tags, and artifacts.

Using the `MlflowClient`, the metadata of the best run is accessed to inspect
the logged information such as model type, dataset version, evaluation
metrics, and hyperparameters. The trained model artifact is then loaded from
MLflow using the run ID.

Finally, the loaded model is used to generate predictions on the test dataset
to verify that it produces the same performance as the original run. This
demonstrates how another engineer can reproduce and reuse a model from the
experiment tracking system.

In [14]:
#  BEGINNER 2D - Retrieve and verify the best model
# ------------------------------------------------------

client = MlflowClient()

# Step 1: Retrieve the run metadata for BEST_RUN_ID
best_run = client.get_run(BEST_RUN_ID)

# Step 2: Print what was logged
if best_run:
    print("Run metadata:")
    print(f"  Model type      : {best_run.data.tags.get('model_type')}")
    print(f"  Dataset version : {best_run.data.tags.get('dataset_version')}")
    print(f"  F1 score        : {best_run.data.metrics.get('f1', 0):.4f}")
    print(f"  Params          : {best_run.data.params}")

# Step 3: Load the model artifact
loaded_model = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")

# Step 4: Verify it produces the same predictions
if loaded_model is not None:
    reloaded_preds = loaded_model.predict(X_test)
    print(f"\nVerification F1: {f1_score(y_test, reloaded_preds):.4f}")
    print("Model successfully retrieved from MLflow")

Run metadata:
  Model type      : RandomForest
  Dataset version : v1
  F1 score        : 0.8747
  Params          : {'model': 'RandomForest', 'n_estimators': '100', 'max_depth': '6'}



Verification F1: 0.8747
Model successfully retrieved from MLflow


## Observation

The best-performing model was successfully retrieved from MLflow using its
stored run ID. The retrieved metadata confirms that the model is a
**RandomForest** trained on dataset version **v1**, with an F1 score of **0.8747**.

The model artifact was loaded from the MLflow tracking server and used to
generate predictions on the test dataset. The verification step produced
the same **F1 score of 0.8747**, confirming that the experiment results are
reproducible and that the model can be reliably retrieved by another
engineer from the MLflow tracking system.

In [15]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert mlflow.get_experiment_by_name("loandefault_prediction") is not None, "Create experiment 'loandefault_prediction'"
    passed.append('MLflow experiment exists')
except Exception as e:
    failed.append(('MLflow experiment exists', str(e)))

try:
    assert len(results) == 3, 'results list should have 3 entries'
    passed.append('3 candidates ran')
except Exception as e:
    failed.append(('3 candidates ran', str(e)))

try:
    assert 'BEST_RUN_ID' in dir() or 'BEST_RUN_ID' in globals(), 'BEST_RUN_ID must be defined'
    passed.append('Best run ID set')
except Exception as e:
    failed.append(('Best run ID set', str(e)))

try:
    assert loaded_model is not None, 'loaded_model should not be None'
    passed.append('Model reloads OK')
except Exception as e:
    failed.append(('Model reloads OK', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')


 Passed: 4/4
  MLflow experiment exists
  3 candidates ran
  Best run ID set
  Model reloads OK

All checks passed!


---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 2

Implement a full experiment comparison pipeline:

**Requirements:**
1. Configure MLflow with the local SQLite URI
2. Write `run_experiment()` that logs: params, 5 metrics (accuracy/precision/recall/f1/auc), model artifact, dataset version tag, and training timestamp tag
3. Run all three candidate models (configurations given in Beginner 2C)
4. Write `find_best_run(experiment_name, metric='f1')` that queries MLflow programmatically (using `MlflowClient.search_runs()`) and returns the run with the highest value of `metric`
5. Load the best model, re-evaluate on test set, and confirm metrics match what was logged
6. **Bonus:** Add a second experiment round where you retrain the best model type on `loans_v2.csv` and log the dataset version change

## Explanation

This advanced implementation builds a complete MLflow experiment tracking
pipeline for model comparison and reproducibility.

The system first configures MLflow to use the local SQLite tracking database.
A reusable `run_experiment()` function is implemented to train models and log
parameters, evaluation metrics, artifacts, and metadata tags such as the
dataset version and training timestamp.

Multiple candidate models are then trained and tracked as separate MLflow
runs. The function `find_best_run()` programmatically queries the experiment
history using the MLflow client and identifies the run with the highest
F1 score.

The best model is retrieved from the tracking server, re-evaluated on the
test dataset, and its metrics are verified against the logged results.
Finally, as a bonus experiment round, the best model type is retrained on
the updated dataset (`loans_v2.csv`) and logged with the new dataset
version for comparison.

In [ ]:
#  ADVANCED - Task 2 Full Implementation
# ----------------------------------------------------------

# YOUR CODE HERE

# Configure MLflow tracking
mlflow.set_tracking_uri(MLFLOW_URI)
experiment = mlflow.set_experiment("loandefault_prediction")
experiment_id = experiment.experiment_id


# ----------------------------------------------------------
# Run a single MLflow experiment
# ----------------------------------------------------------
def run_experiment(model, model_name, params, X_tr, X_te, y_tr, y_te, dataset_version):

    with mlflow.start_run(run_name=model_name) as run:

        # Train model
        model.fit(X_tr, y_tr)

        # Predictions
        preds = model.predict(X_te)
        proba = model.predict_proba(X_te)[:, 1]

        # Metrics
        acc  = accuracy_score(y_te, preds)
        prec = precision_score(y_te, preds)
        rec  = recall_score(y_te, preds)
        f1   = f1_score(y_te, preds)
        auc  = roc_auc_score(y_te, proba)

        # Log parameters
        mlflow.log_params(params)

        # Log metrics
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("auc", auc)

        # Log tags
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("model_type", model_name)
        mlflow.set_tag("training_timestamp", datetime.now().isoformat())

        # Log model artifact
        mlflow.sklearn.log_model(model, "model")

        return run.info.run_id


# ----------------------------------------------------------
# Candidate model configurations
# ----------------------------------------------------------
candidates = [
    (
        LogisticRegression(C=0.5, max_iter=1000, random_state=42),
        "LogisticRegression",
        {"model": "LogisticRegression", "C": 0.5, "max_iter": 1000}
    ),
    (
        RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        "RandomForest",
        {"model": "RandomForest", "n_estimators": 100, "max_depth": 6}
    ),
    (
        GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                   max_depth=4, random_state=42),
        "GradientBoosting",
        {"model": "GradientBoosting", "n_estimators": 100,
         "learning_rate": 0.1, "max_depth": 4}
    )
]


# ----------------------------------------------------------
# Run experiments
# ----------------------------------------------------------
results = []

for model, name, params in candidates:

    run_id = run_experiment(
        model,
        name,
        params,
        X_train,
        X_test,
        y_train,
        y_test,
        dataset_version="v1"
    )

    f1 = mlflow.get_run(run_id).data.metrics.get("f1")

    results.append({
        "name": name,
        "run_id": run_id,
        "f1": f1
    })


# ----------------------------------------------------------
# Find best run programmatically
# ----------------------------------------------------------
def find_best_run(experiment_name, metric="f1"):

    client = MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)

    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=[f"metrics.{metric} DESC"]
    )

    return runs[0]


best_run = find_best_run("loandefault_prediction", "f1")

print("="*55)
print("Best run identified")
print("="*55)
print("Run ID:", best_run.info.run_id)
print("Model :", best_run.data.tags.get("model_type"))
print("F1    :", best_run.data.metrics.get("f1"))


# ----------------------------------------------------------
# Load best model and verify metrics
# ----------------------------------------------------------
best_model = mlflow.sklearn.load_model(f"runs:/{best_run.info.run_id}/model")

preds = best_model.predict(X_test)
proba = best_model.predict_proba(X_test)[:, 1]

print("\nVerification metrics")
print("F1 :", f1_score(y_test, preds))
print("AUC:", roc_auc_score(y_test, proba))


# ----------------------------------------------------------
# BONUS: retrain best model type on loans_v2.csv
# ----------------------------------------------------------
df_v2 = pd.read_csv(LAB_DIR / "data" / "loans_v2.csv").dropna()

X_v2 = df_v2[FEATURES]
y_v2 = df_v2[TARGET]

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2, y_v2, test_size=0.2, random_state=42
)

best_model_type = best_run.data.tags.get("model_type")

if best_model_type == "RandomForest":
    model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)

elif best_model_type == "GradientBoosting":
    model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                       max_depth=4, random_state=42)

else:
    model = LogisticRegression(C=0.5, max_iter=1000, random_state=42)

print("\nRunning second experiment round on dataset v2")

new_run_id = run_experiment(
    model,
    f"{best_model_type}_retrain_v2",
    {"model": best_model_type},
    X_train_v2,
    X_test_v2,
    y_train_v2,
    y_test_v2,
    dataset_version="v2"
)

print("\nNew run logged for dataset v2")
print(f"Run ID: {new_run_id}")

2026/03/13 14:20:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 14:20:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/13 14:20:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 14:20:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Best run identified
Run ID: 6abbd6b370504a16b4684084bf68ff5b
Model : RandomForest
F1    : 0.8747433264887063



Verification metrics
F1 : 0.8747433264887063
AUC: 0.9296354166666667

Running second experiment round on dataset v2


2026/03/13 14:20:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 14:20:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


## Observation

The MLflow client successfully identified the best experiment run, which
corresponds to the **RandomForest model** with an F1 score of **0.8747**.
The model artifact was retrieved from the tracking server and re-evaluated
on the test dataset, producing the same F1 and AUC scores, confirming that
the experiment results are reproducible.

In the bonus experiment round, the best model type was retrained using the
updated dataset **loans_v2.csv**, and the new run was logged in MLflow with
the dataset version tag **v2**. The run ID confirms that the new experiment
was successfully recorded.

During execution, MLflow displayed warnings related to deprecated arguments
and model serialization methods. These warnings are informational and do
not affect the correctness of the experiment runs in this lab environment.

---
# TASK 3 - Model Versioning
> Give your model a name that tells the whole story
---

##  Task 3 - Model Versioning - Naming, Metadata & Semantic Versions

### Scenario
> LoanGuard's best model from Task 2 needs to be packaged for the team. The previous engineer just saved files as `model.pkl` with no metadata. When it broke, nobody knew what version was running or how to go back. You need to create a versioned model artifact with full metadata and implement semantic versioning logic.

### Objective
By the end of this task you will be able to:
- Save a model artifact with complete metadata (dataset version, metrics, git hash, author)
- Implement semantic versioning logic (MAJOR.MINOR.PATCH) for model releases
- Register a named, versioned model in MLflow Model Registry
- Retrieve a specific model version by name

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  3A - Save a model with complete metadata

Save the best model from Task 2 alongside a metadata sidecar file that fully describes it.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Save the model with `pickle.dump()`. Create a separate `_metadata.json` file in the same folder with all relevant context.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Metadata should include: model_name, version, dataset_version, metrics, params, trained_at, and a simulated git_hash (`hashlib.md5(str(time.time()).encode()).hexdigest()[:8]`).

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def save_versioned_model(model, name, version, metrics, params, dataset_version):
    path = LAB_DIR / 'models' / f'{name}-{version}'
    path.mkdir(exist_ok=True)
    with open(path / 'model.pkl', 'wb') as f:
        pickle.dump(model, f)
    meta = {
        'name': name, 'version': version,
        'dataset_version': dataset_version,
        'metrics': metrics, 'params': params,
        'git_hash': hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        'trained_at': datetime.now().isoformat(),
        'mlflow_run_id': BEST_RUN_ID
    }
    with open(path / 'metadata.json', 'w') as f:
        json.dump(meta, f, indent=2)
    return path, meta
```

</details>

## Explanation

In this step, the best-performing model from Task 2 is packaged as a
versioned model artifact. The model is saved together with a metadata
sidecar file that records all relevant context required to reproduce
the model.

The function creates a versioned directory named using the model name
and semantic version (e.g., `loan-default-detector-v1.0.0`). Inside this
directory, the trained model is serialized using `pickle`, and a
`metadata.json` file is created containing information such as dataset
version, evaluation metrics, hyperparameters, training timestamp,
MLflow run ID, and a simulated git commit hash.

This approach ensures that each model artifact is fully traceable and
can be linked back to the experiment run that produced it.

In [ ]:
#  BEGINNER 3A - Save versioned model with metadata
# ------------------------------------------------------------

def save_versioned_model(model, name, version, metrics, params,
                          dataset_version, run_id):
    """
    Save a model and its complete metadata sidecar.

    Creates:
      models/{name}-{version}/
        model.pkl        ← serialised model
        metadata.json    ← full provenance record

    Returns
    -------
    (path : Path, metadata : dict)
    """
    # Step 1: Create output directory
    path = LAB_DIR / "models" / f"{name}-{version}"
    path.mkdir(exist_ok=True)

    # Step 2: Save model with pickle
    with open(path / "model.pkl", "wb") as f:
        pickle.dump(model, f)

    # Step 3: Build metadata dict
    metadata = {
        "name":             name,
        "version":          version,
        "dataset_version":  dataset_version,
        "metrics":          metrics,
        "params":           params,
        "git_hash":         hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        "trained_at":       datetime.now().isoformat(),
        "mlflow_run_id":    run_id,
        "author":           "loanGuard-ml-team"
    }

    # Step 4: Save metadata as JSON
    with open(path / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    return path, metadata


# ----- Save the best model from Task 2 -----
best_run_data   = MlflowClient().get_run(BEST_RUN_ID)
best_metrics    = best_run_data.data.metrics
best_params     = best_run_data.data.params
best_fitted     = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")

model_path, model_meta = save_versioned_model(
    model           = best_fitted,
    name            = "loan-default-detector",
    version         = "v1.0.0",
    metrics         = best_metrics,
    params          = best_params,
    dataset_version = "v1",
    run_id          = BEST_RUN_ID
)

print(f"Saved to: {model_path}")
print("\nMetadata:")
print(json.dumps(model_meta, indent=2))

## Observation

The best-performing model from Task 2 was successfully saved as a
versioned model artifact under the directory
`loan-default-detector-v1.0.0`. The trained model was serialized as
`model.pkl`, and a corresponding `metadata.json` file was created to
store full provenance information.

The metadata records important context including the dataset version,
evaluation metrics, model parameters, MLflow run ID, training timestamp,
and a simulated git hash. This ensures that the model artifact is fully
traceable and can be reproduced or audited in the future.

During execution, MLflow briefly downloaded the stored model artifact
from the tracking server before saving it locally. This is expected
behavior when retrieving models from MLflow.

###  3B - Semantic versioning logic

Implement a function that determines the correct next version number given the type of change.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Parse the version string by splitting on '.': `major, minor, patch = version.lstrip('v').split('.')`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

For 'major' bump: increment major, reset minor and patch to 0. For 'minor': increment minor, reset patch. For 'patch': just increment patch.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def bump_version(current, bump_type):
    major, minor, patch = [int(x) for x in current.lstrip('v').split('.')]
    if bump_type == 'major':   return f'v{major+1}.0.0'
    elif bump_type == 'minor': return f'v{major}.{minor+1}.0'
    elif bump_type == 'patch': return f'v{major}.{minor}.{patch+1}'
```

</details>

## Explanation

This step implements **semantic versioning** logic for model releases.
Semantic versioning follows the format **MAJOR.MINOR.PATCH** and helps
teams clearly communicate the impact of model updates.

A **major version bump** indicates a breaking change such as switching
to a completely different model architecture. A **minor version bump**
represents a backward-compatible improvement such as retraining with
additional features or new data. A **patch version bump** represents
small adjustments such as tuning thresholds or fixing minor issues.

The `bump_version()` function parses the current version string,
applies the appropriate increment based on the type of change, and
returns the updated semantic version.

In [ ]:
#  BEGINNER 3B - Semantic versioning
# ------------------------------------------------------------

def bump_version(current_version, bump_type):
    """
    Increment a semantic version string.

    Parameters
    ----------
    current_version : str  e.g. 'v1.2.3'
    bump_type       : str  one of 'major', 'minor', 'patch'

    Returns
    -------
    str  e.g. 'v2.0.0' for a major bump on 'v1.2.3'

    Rules
    -----
    major → increment MAJOR, reset MINOR and PATCH to 0
    minor → increment MINOR, reset PATCH to 0
    patch → increment PATCH only
    """
    # Step 1: Parse current_version into major, minor, patch integers
    major, minor, patch = [int(x) for x in current_version.lstrip('v').split('.')]

    # Step 2: Apply bump logic
    if bump_type == "major":
        major += 1
        minor = 0
        patch = 0
    elif bump_type == "minor":
        minor += 1
        patch = 0
    elif bump_type == "patch":
        patch += 1

    return f"v{major}.{minor}.{patch}"


# ----- Test all bump types -----
start = "v1.0.0"
print(f"Start version      : {start}")
print(f"After patch bump   : {bump_version(start, 'patch')}   ← threshold tweak")
print(f"After minor bump   : {bump_version(start, 'minor')}   ← new training data")
print(f"After major bump   : {bump_version(start, 'major')}   ← new architecture")
print()

# Simulate LoanGuard's version history
history = [
    ("v1.0.0", "Initial release - Random Forest on loans_v1"),
    (bump_version("v1.0.0", "patch"), "Fixed decision threshold from 0.5 → 0.46"),
    (bump_version("v1.0.1", "minor"), "Retrained on loans_v2 with new num_accounts feature"),
    (bump_version("v1.1.0", "major"), "Switched to GradientBoosting - breaking output format change"),
]
print("LoanGuard model version history:")
for ver, desc in history:
    print(f"  {ver:<12} {desc}")

## Observation

The semantic versioning function successfully generated new model
versions based on the type of change applied. A **patch bump**
incremented only the patch number (v1.0.0 → v1.0.1), a **minor bump**
incremented the minor version and reset the patch number (v1.0.0 → v1.1.0),
and a **major bump** incremented the major version while resetting the
minor and patch numbers (v1.0.0 → v2.0.0).

The simulated version history demonstrates how semantic versioning
communicates the impact of changes to a model, distinguishing between
small fixes, dataset updates, and major architectural changes.

###  3C - Register in MLflow Model Registry

Register your best model in the MLflow Model Registry under a named model, then retrieve it by name.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `mlflow.register_model(model_uri, name)` where `model_uri = f'runs:/{run_id}/model'`.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

After registration, use `MlflowClient().get_registered_model(name)` to inspect it. `client.get_latest_versions(name)` lists all versions.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
model_uri = f"runs:/{BEST_RUN_ID}/model"
reg = mlflow.register_model(model_uri, "LoanDefaultDetector")
print(reg.version, reg.name)

client  = MlflowClient()
version = client.get_latest_versions("LoanDefaultDetector")[0]
loaded  = mlflow.sklearn.load_model(f"models:/LoanDefaultDetector/{version.version}")
```

</details>

## Explanation

In this step, the best-performing model from the MLflow experiment is
registered in the **MLflow Model Registry** under a named model. The
registry allows teams to track different versions of a model and manage
its lifecycle (development, staging, production).

First, the model artifact associated with the best run ID is registered
in the registry using a model name. MLflow automatically assigns a
version number for the registered model.

Next, all versions of the registered model are retrieved using the
MLflow client so the available model versions can be inspected. Finally,
the model is loaded from the registry by name and version to verify that
it produces the same predictions and evaluation metrics as the original
experiment run.

In [ ]:
#  BEGINNER 3C - Register model in MLflow Registry
# ----------------------------------------------------------

REGISTRY_NAME = "LoanDefaultDetector"

# Step 1: Register the best model run in the MLflow registry
registered = mlflow.register_model(
    model_uri=f"runs:/{BEST_RUN_ID}/model",
    name=REGISTRY_NAME
)

if registered:
    print(f"Registered model : {registered.name}")
    print(f"Version          : {registered.version}")
    print(f"Source run       : {registered.source}")

# Step 2: Retrieve and inspect all registered versions
client = MlflowClient()
versions = client.search_model_versions(f"name='{REGISTRY_NAME}'")

print("\nRegistered model versions:")
for v in versions:
    print(f"  Version {v.version}  |  Run ID: {v.run_id}  |  Stage: {v.current_stage}")

# Step 3: Load the registered model by name and verify
registry_model = mlflow.sklearn.load_model(f"models:/{REGISTRY_NAME}/{registered.version}")

if registry_model is not None:
    verify_preds = registry_model.predict(X_test)
    print(f"\nVerification F1 from registry: {f1_score(y_test, verify_preds):.4f}")
    print(" Model successfully retrieved from registry by name")

## Observation

The best-performing model from Task 2 was successfully registered in the
MLflow Model Registry under the name **LoanDefaultDetector**. MLflow
automatically created **version 1** of the model and linked it to the
corresponding experiment run.

The registry was then queried to list all available versions of the
registered model. Finally, the model was loaded from the registry using
its name and version, and predictions were generated on the test dataset.

The verification step produced the same **F1 score of 0.8747**, confirming
that the model retrieved from the registry behaves identically to the
original experiment artifact.

During registration, MLflow displayed an informational warning related to
artifact paths. This does not affect the correctness of the registration
process.

In [ ]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert (LAB_DIR / 'models' / 'loan-default-detector-v1.0.0' / 'model.pkl').exists(), 'model.pkl not found'
    passed.append('Versioned model saved')
except Exception as e:
    failed.append(('Versioned model saved', str(e)))

try:
    assert (LAB_DIR / 'models' / 'loan-default-detector-v1.0.0' / 'metadata.json').exists(), 'metadata.json not found'
    passed.append('Metadata saved')
except Exception as e:
    failed.append(('Metadata saved', str(e)))

try:
    assert bump_version('v1.0.0', 'patch') == 'v1.0.1', 'patch bump failed'
    passed.append('Patch bump correct')
except Exception as e:
    failed.append(('Patch bump correct', str(e)))

try:
    assert bump_version('v1.0.0', 'minor') == 'v1.1.0', 'minor bump failed'
    passed.append('Minor bump correct')
except Exception as e:
    failed.append(('Minor bump correct', str(e)))

try:
    assert bump_version('v1.0.0', 'major') == 'v2.0.0', 'major bump failed'
    passed.append('Major bump correct')
except Exception as e:
    failed.append(('Major bump correct', str(e)))

try:
    assert registered is not None, 'registered model should not be None'
    passed.append('Model registered')
except Exception as e:
    failed.append(('Model registered', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')

---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 3

Build a complete model versioning and registry system:

**Requirements:**
1. `save_versioned_model()` - saves model + full metadata sidecar (all fields from Beginner 3A)
2. `bump_version()` - semantic version logic with validation (reject unknown bump types)
3. `ModelVersionHistory` class with methods:
   - `record(version, change_type, description, metrics)` - adds an entry
   - `get_changelog()` - returns formatted changelog string
   - `get_version_for_rollback(n_versions_back)` - returns the version string from n releases ago
4. Register model in MLflow Registry, add a description, and add the semantic version as a tag
5. **Demonstrate rollback:** Simulate a production failure, use `get_version_for_rollback(1)` to identify the previous version, load it from the local store, and confirm it produces different (better) metrics than the failed version

## Explanation

This implementation builds a complete model versioning and registry system
for managing machine learning model releases.

First, the trained model is saved together with a metadata sidecar file that
captures full provenance information including dataset version, metrics,
parameters, training timestamp, git hash, and MLflow run ID.

A semantic versioning function is implemented to manage version increments
using the MAJOR.MINOR.PATCH scheme while validating allowed bump types.

A `ModelVersionHistory` class tracks the lifecycle of model releases,
allowing version history recording, changelog generation, and rollback
version retrieval.

The model is then registered in the MLflow Model Registry with a semantic
version tag and description. Finally, a rollback scenario is simulated
where a failed production model triggers loading a previous version from
the local version store to restore stable performance.

In [ ]:
# ADVANCED - Task 3 Full Implementation
# ----------------------------------------------------------

# ----------------------------------------------------------
# 1. Save model with metadata sidecar
# ----------------------------------------------------------

def save_versioned_model(model, name, version, metrics, params, dataset_version, run_id):

    path = LAB_DIR / "models" / f"{name}-{version}"
    path.mkdir(exist_ok=True)

    # Save model artifact
    with open(path / "model.pkl", "wb") as f:
        pickle.dump(model, f)

    # Metadata
    metadata = {
        "name": name,
        "version": version,
        "dataset_version": dataset_version,
        "metrics": metrics,
        "params": params,
        "git_hash": hashlib.md5(str(time.time()).encode()).hexdigest()[:8],
        "trained_at": datetime.now().isoformat(),
        "mlflow_run_id": run_id,
        "author": "loanGuard-ml-team"
    }

    with open(path / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    return path, metadata


# ----------------------------------------------------------
# 2. Semantic versioning with validation
# ----------------------------------------------------------

def bump_version(current_version, bump_type):

    if bump_type not in ["major", "minor", "patch"]:
        raise ValueError(f"Unknown bump type: {bump_type}")

    major, minor, patch = [int(x) for x in current_version.lstrip("v").split(".")]

    if bump_type == "major":
        major += 1
        minor = 0
        patch = 0

    elif bump_type == "minor":
        minor += 1
        patch = 0

    elif bump_type == "patch":
        patch += 1

    return f"v{major}.{minor}.{patch}"


# ----------------------------------------------------------
# 3. Model version history tracker
# ----------------------------------------------------------

class ModelVersionHistory:

    def __init__(self):
        self.history = []

    def record(self, version, change_type, description, metrics):

        entry = {
            "version": version,
            "change_type": change_type,
            "description": description,
            "metrics": metrics,
            "timestamp": datetime.now().isoformat()
        }

        self.history.append(entry)

    def get_changelog(self):

        lines = ["LoanGuard Model Changelog:"]
        for entry in self.history:
            lines.append(
                f"{entry['version']} ({entry['change_type']}) - "
                f"{entry['description']} | F1: {entry['metrics'].get('f1',0):.4f}"
            )

        return "\n".join(lines)

    def get_version_for_rollback(self, n_versions_back):

        if n_versions_back >= len(self.history):
            raise ValueError("Rollback exceeds available history")

        return self.history[-(n_versions_back+1)]["version"]


# ----------------------------------------------------------
# Prepare version history tracker
# ----------------------------------------------------------

history = ModelVersionHistory()


# ----------------------------------------------------------
# Save current best model (v1.0.0)
# ----------------------------------------------------------

best_run_data = MlflowClient().get_run(BEST_RUN_ID)
best_metrics  = best_run_data.data.metrics
best_params   = best_run_data.data.params

best_model = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")

path_v1, meta_v1 = save_versioned_model(
    best_model,
    "loan-default-detector",
    "v1.0.0",
    best_metrics,
    best_params,
    "v1",
    BEST_RUN_ID
)

history.record(
    "v1.0.0",
    "major",
    "Initial production release",
    best_metrics
)


# ----------------------------------------------------------
# Simulate a new patch version (v1.0.1)
# ----------------------------------------------------------

new_version = bump_version("v1.0.0", "patch")

path_v2, meta_v2 = save_versioned_model(
    best_model,
    "loan-default-detector",
    new_version,
    best_metrics,
    best_params,
    "v1",
    BEST_RUN_ID
)

history.record(
    new_version,
    "patch",
    "Threshold tuning for improved recall",
    best_metrics
)


# ----------------------------------------------------------
# 4. Register model in MLflow registry
# ----------------------------------------------------------

REGISTRY_NAME = "LoanDefaultDetector"

registered = mlflow.register_model(
    model_uri=f"runs:/{BEST_RUN_ID}/model",
    name=REGISTRY_NAME
)

client = MlflowClient()

client.update_registered_model(
    name=REGISTRY_NAME,
    description="Loan default prediction model for LoanGuard risk platform"
)

client.set_model_version_tag(
    name=REGISTRY_NAME,
    version=registered.version,
    key="semantic_version",
    value="v1.0.0"
)

print("Model registered in MLflow registry")
print("Version:", registered.version)


# ----------------------------------------------------------
# Print changelog
# ----------------------------------------------------------

print("\nModel changelog:")
print(history.get_changelog())


# ----------------------------------------------------------
# 5. Demonstrate rollback
# ----------------------------------------------------------

print("\nSimulating production failure...")

failed_version = "v1.0.1"

rollback_version = history.get_version_for_rollback(1)

print("Failed version :", failed_version)
print("Rolling back to:", rollback_version)


# Load rollback model from local store
rollback_path = LAB_DIR / "models" / f"loan-default-detector-{rollback_version}" / "model.pkl"

with open(rollback_path, "rb") as f:
    rollback_model = pickle.load(f)

rollback_preds = rollback_model.predict(X_test)

print("\nRollback model F1:",
      f1_score(y_test, rollback_preds))

## Observation

The advanced model versioning system successfully saved the trained model
with full metadata and applied semantic versioning to track model releases.
Two versions were recorded in the version history: **v1.0.0** (initial
production release) and **v1.0.1** (patch update for threshold tuning).

The model was registered in the **MLflow Model Registry**, where a new
version of the registered model was created and linked to the experiment
run.

A rollback scenario was then simulated to represent a production failure.
Using the version history tracker, the system identified the previous
stable version (**v1.0.0**) and loaded it from the local model store.
The rollback model produced the same **F1 score of 0.8747**, confirming
that the system can reliably restore a previous model version when needed.

During execution, MLflow displayed informational warnings related to
artifact paths and model registration. These warnings do not affect the
correctness of the versioning or registry workflow.

---
# TASK 4 - Registry Lifecycle Management
> Staging → Production → Archived: govern your models
---

##  Task 4 - Registry Lifecycle - Staging, Production, Archived

### Scenario
> LoanGuard just hired a new senior ML engineer. She asks: 'What model is in production right now? When was it approved? Who approved it? What does rollback look like?' Nobody can answer. In this task you'll implement the full governance lifecycle - every model moves through defined stages with approvals and audit logs.

### Objective
By the end of this task you will be able to:
- Transition a model through Staging → Production → Archived stages in MLflow
- Write an audit log that records every lifecycle event
- Implement a rollback function that demotes the current production model
- Query the registry to answer governance questions programmatically

---

###  Choose Your Path

| Path | Description |
|------|-------------|
|  **Beginner** | Scaffolded code - fill in the blanks (`# YOUR CODE HERE`) |
|  **Advanced** | Open-ended - write from scratch using only the requirements |

**You are on the  Beginner path.** Skip to the  Advanced section if you want a challenge.

---

###  4A - Transition model through lifecycle stages

Move the registered model through the stages: None → Staging → Production

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

Use `MlflowClient().transition_model_version_stage(name, version, stage)` where stage is 'Staging', 'Production', or 'Archived'.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

After each transition, add a comment: `client.update_model_version(name, version, description=...)` to record the reason.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
client = MlflowClient()
client.transition_model_version_stage(
    name    = REGISTRY_NAME,
    version = "1",
    stage   = "Staging",
    archive_existing_versions=False
)
```

</details>

## Explanation

This step implements the lifecycle governance of a registered model
within the MLflow Model Registry. Models move through defined stages
such as **Staging**, **Production**, and **Archived** to represent their
deployment readiness and operational status.

The `transition_stage()` function uses the MLflow client to change the
stage of a specific model version. It also updates the model version
description to record the reason for the transition, creating a simple
audit trail.

In this scenario, the model is first promoted to **Staging** after
passing evaluation tests, and then promoted to **Production** once it
receives approval for deployment. The registry is then queried to
confirm the model's current stage and recorded description.

In [ ]:
#  BEGINNER 4A - Lifecycle stage transitions
# ---------------------------------------------------------

client = MlflowClient()

def transition_stage(name, version, stage, reason=""):
    """
    Transition a registered model version to a new stage and record the reason.

    Parameters
    ----------
    name    : str  registered model name
    version : str  model version number as string e.g. '1'
    stage   : str  'Staging', 'Production', or 'Archived'
    reason  : str  human-readable reason for the transition
    """
    # Step 1: Transition the model stage
    client.transition_model_version_stage(
        name=name,
        version=version,
        stage=stage
    )

    # Step 2: Update the model version description with the reason
    client.update_model_version(
        name=name,
        version=version,
        description=reason
    )

    print(f"  {name} v{version}  →  {stage}")
    if reason:
        print(f"Reason: {reason}")


# ----- Walk the model through its lifecycle -----
latest = client.get_latest_versions(REGISTRY_NAME)
if latest:
    ver = latest[0].version

    print("Stage transitions for LoanDefaultDetector:")
    print("─" * 50)

    transition_stage(
        REGISTRY_NAME, ver, "Staging",
        reason="Passed unit tests and offline evaluation - F1 > 0.72 threshold met"
    )

    # YOUR CODE HERE - transition to Production with a reason
    transition_stage(
        REGISTRY_NAME, ver, "Production",
        reason="Approved by senior ML engineer after staging validation and governance review"
    )

    # Verify current stage
    current = client.get_model_version(REGISTRY_NAME, ver)
    print(f"\nCurrent stage: {current.current_stage}")
    print(f"Description  : {current.description}")

## Observation

The registered model **LoanDefaultDetector v5** was successfully
transitioned through the lifecycle stages in the MLflow Model Registry.
The model was first promoted to **Staging** after passing unit tests and
offline evaluation criteria. It was then promoted to **Production**
following approval from the senior ML engineer.

The lifecycle transition also recorded the approval reason in the model
version description, creating a simple audit trail for governance.
Querying the registry confirmed that the model is currently in the
**Production** stage and that the approval description has been stored
correctly.

###  4B - Build an audit log

Every lifecycle event should be recorded. Implement an audit log that tracks: timestamp, model name, version, old stage, new stage, and the actor/reason.

## Explanation

This step implements an **audit logging system** for model lifecycle
events. Each time a model transitions between registry stages, the event
is recorded with full governance metadata including timestamp, model
name, version, previous stage, new stage, the actor responsible for the
change, and the reason for the transition.

The audit log is stored both **in memory** (for immediate inspection) and
persisted to disk in a JSON file to maintain a permanent governance
record. This ensures that model lifecycle actions are traceable and can
be reviewed later for compliance, debugging, or rollback analysis.

In [ ]:
#  BEGINNER 4B - Audit log
# ---------------------------------------------------------

AUDIT_LOG = []   # list of event dicts

def log_audit_event(model_name, version, from_stage, to_stage,
                     actor, reason):
    """
    Record a lifecycle event in the audit log and persist to disk.
    """
    event = {
        "timestamp":    datetime.now().isoformat(),
        "model_name":   model_name,
        "version":      version,
        "from_stage":   from_stage,
        "to_stage":     to_stage,
        "actor":        actor,
        "reason":       reason
    }

    # Step 1: Append to in-memory log
    AUDIT_LOG.append(event)

    # Step 2: Save to JSON file (append-style - read existing, add, write back)
    log_path = LAB_DIR / "audit_log.json"

    if log_path.exists():
        with open(log_path, "r") as f:
            existing = json.load(f)
    else:
        existing = []

    existing.append(event)

    with open(log_path, "w") as f:
        json.dump(existing, f, indent=2)

    return event


def print_audit_log():
    """Print a formatted audit log table."""
    if not AUDIT_LOG:
        print("Audit log is empty")
        return
    print(f"{'Timestamp':<26} {'Model':<22} {'v':<4} {'From':<12} {'To':<12} {'Actor':<18} Reason")
    print("─" * 110)
    for e in AUDIT_LOG:
        ts = e['timestamp'][:19]
        print(f"{ts:<26} {e['model_name']:<22} {e['version']:<4} "
              f"{e['from_stage']:<12} {e['to_stage']:<12} {e['actor']:<18} {e['reason']}")


# ----- Replay lifecycle events through audit log -----
ver = client.get_latest_versions(REGISTRY_NAME)[0].version

log_audit_event(REGISTRY_NAME, ver, "None", "Staging",
                "ci-system", "Automated: offline eval passed F1 threshold")
log_audit_event(REGISTRY_NAME, ver, "Staging", "Production",
                "ml-lead@loanGuard.ai", "Manual approval: A/B test shows +3% approval lift")

print("LoanGuard Model Audit Log:")
print("=" * 110)
print_audit_log()

## Observation

The audit logging system successfully recorded lifecycle events for the
registered model **LoanDefaultDetector v7**. Each transition event
captured important governance metadata including timestamp, model name,
version, previous stage, new stage, actor responsible for the change,
and the reason for the transition.

Two lifecycle events were recorded in the audit log: the automated
promotion from **None → Staging** after offline evaluation passed the
required threshold, and the manual approval from **Staging → Production**
by the ML lead after successful A/B testing.

The audit log provides a transparent governance trail that allows teams
to trace model deployment decisions and verify who approved each stage
transition.

###  4C - Rollback

Simulate a production failure: a new model version is promoted, starts failing, and you need to roll back to the previous production model.

<details>
<summary> Hint 1 - Click to reveal (try first!)</summary>

To rollback: archive the current production version, then transition the previous version back to Production.

</details>

<details>
<summary> Hint 2 - A bit more help</summary>

Use `client.search_model_versions(f"name='{name}'")` to find all versions. Filter by `current_stage == 'Archived'` to find previous production candidates.

</details>

<details>
<summary> Hint 3 - Nearly the answer</summary>

```python
def rollback_production(name, rollback_to_version, reason):
    # 1. Find and archive current production version
    current_prod = [v for v in client.search_model_versions(f"name='{name}'")
                    if v.current_stage == 'Production']
    if current_prod:
        client.transition_model_version_stage(name, current_prod[0].version, 'Archived')
    # 2. Promote rollback target to Production
    client.transition_model_version_stage(name, rollback_to_version, 'Production')
```

</details>

## Explanation

This step simulates a real-world production failure and demonstrates how
model rollback is performed using the MLflow Model Registry.

A deliberately weak model is first trained and mistakenly promoted to
**Production**, representing a faulty deployment. The `rollback_production()`
function then restores the previously stable model version.

The rollback procedure performs three governance actions:
1. Identify the current production model version.
2. Move that failing version to the **Archived** stage.
3. Promote the previous stable version back to **Production**.

Each lifecycle change is recorded in the audit log to maintain a
complete governance trail of deployment and rollback events.

In [ ]:
#  BEGINNER 4C - Simulate promotion of v2, then rollback
# ---------------------------------------------------------

# First: Register a second (worse) model version to simulate a bad deployment
print("Simulating a bad deployment...")

# Train a deliberately weak model
weak_model = LogisticRegression(C=0.001, max_iter=100, random_state=42)
with mlflow.start_run(run_name="WeakModel_bad_deploy") as bad_run:
    weak_model.fit(X_train, y_train)
    weak_preds = weak_model.predict(X_test)
    mlflow.log_metric("f1", f1_score(y_test, weak_preds))
    mlflow.set_tag("dataset_version", "v1")
    mlflow.sklearn.log_model(weak_model, "model")
    bad_run_id = bad_run.info.run_id

# Register the weak model as version 2
bad_registered = mlflow.register_model(f"runs:/{bad_run_id}/model", REGISTRY_NAME)
bad_version = bad_registered.version

# Promote bad model to Production (simulating a mistaken deployment)
client.transition_model_version_stage(
    REGISTRY_NAME,
    bad_version,
    "Production",
    archive_existing_versions=True
)

log_audit_event(
    REGISTRY_NAME,
    bad_version,
    "Staging",
    "Production",
    "auto-deploy-pipeline",
    "Automated promotion - metric check bypassed (BUG)"
)

print(f"Bad model (version {bad_version}) promoted to Production - F1: {f1_score(y_test, weak_preds):.4f}")
print("  Production is now degraded!")

print()
print("─" * 50)
print("Initiating rollback...")
print("─" * 50)


def rollback_production(registry_name, rollback_to_version, reason, actor):
    """
    Roll back production to a previously archived version.

    Steps:
      1. Find the current Production version and move it to Archived
      2. Move rollback_to_version back to Production
      3. Log both events in the audit log
    """

    # Step 1: Find current production version
    current_prod_versions = client.get_latest_versions(
        registry_name,
        stages=["Production"]
    )

    # Step 2: Archive current production
    for v in current_prod_versions:

        client.transition_model_version_stage(
            name=registry_name,
            version=v.version,
            stage="Archived"
        )

        log_audit_event(
            registry_name,
            v.version,
            "Production",
            "Archived",
            actor,
            f"ROLLBACK: replaced by v{rollback_to_version}. {reason}"
        )

    # Step 3: Promote rollback target back to Production
    client.transition_model_version_stage(
        name=registry_name,
        version=rollback_to_version,
        stage="Production"
    )

    log_audit_event(
        registry_name,
        rollback_to_version,
        "Archived",
        "Production",
        actor,
        f"ROLLBACK TARGET: {reason}"
    )

    print(f" Rolled back to version {rollback_to_version}")


# Execute rollback to version 1
v1_version = client.get_latest_versions(REGISTRY_NAME, stages=["Archived"])

if v1_version:
    rollback_production(
        registry_name=REGISTRY_NAME,
        rollback_to_version=v1_version[0].version,
        reason="Production F1 dropped from 0.72 to 0.41 after v2 deployment",
        actor="ml-lead@loanGuard.ai"
    )

print()
print("Final Audit Log:")
print("=" * 110)
print_audit_log()

## Observation

The rollback simulation successfully demonstrated how a faulty production
deployment can be recovered using the MLflow Model Registry lifecycle
controls. A deliberately weak model (**version 12**) was promoted to the
Production stage, resulting in degraded performance (F1 ≈ 0.6973).

The rollback procedure identified the previous stable version
(**version 11**) and restored it to the Production stage while moving the
faulty model to the Archived stage. All lifecycle events were recorded in
the audit log, including the automated faulty deployment, the rollback
decision, and the restoration of the stable model.

This workflow confirms that the registry lifecycle management system can
safely recover from production failures while maintaining a complete
governance trail of deployment and rollback actions.

###  4D - Governance Query

Answer the governance questions that LoanGuard's new engineer asked - programmatically.

In [ ]:
#  BEGINNER 4D - Answer governance questions programmatically
# ------------------------------------------------------------

print("╔══════════════════════════════════════════════════════╗")
print("║        LoanGuard Model Governance Report             ║")
print("╚══════════════════════════════════════════════════════╝")
print()

# Q1: What model is in production right now?
# YOUR CODE HERE - query the registry and print name + version + stage
prod_models = client.get_latest_versions(REGISTRY_NAME, stages=["Production"])

print(f"Q1 - Current production model:")
for m in prod_models:
    print(f"     {m.name}  version {m.version}  ({m.current_stage})")

print()

# Q2: When was it promoted to production?
# Hint: look in AUDIT_LOG for the most recent 'Production' event
# YOUR CODE HERE
prod_event = None
for e in reversed(AUDIT_LOG):
    if e["to_stage"] == "Production":
        prod_event = e
        break

if prod_event:
    print(f"Q2 - Promoted to production at: {prod_event['timestamp']}")
else:
    print("Q2 - Promoted to production at: Unknown")

print()

# Q3: Who approved it?
# Hint: look for actor in the Production audit event
# YOUR CODE HERE
if prod_event:
    print(f"Q3 - Approved by: {prod_event['actor']}")
else:
    print("Q3 - Approved by: Unknown")

print()

# Q4: What data trained it?
# Hint: use MlflowClient to get the run that produced it, then check tags
# YOUR CODE HERE
if prod_models:
    prod_run = client.get_run(prod_models[0].run_id)
    dataset_version = prod_run.data.tags.get("dataset_version", "Unknown")
    print(f"Q4 - Training dataset: {dataset_version}")
else:
    print("Q4 - Training dataset: Unknown")

print()

# Q5: What were the production model's metrics?
# YOUR CODE HERE
if prod_models:
    metrics = prod_run.data.metrics
    print(f"Q5 - Metrics at registration:")
    for k, v in metrics.items():
        print(f"     {k}: {v:.4f}")
else:
    print("Q5 - Metrics at registration: Unknown")

## Observation

The governance query system successfully retrieved key information about
the currently deployed model from the MLflow Model Registry and audit
log. The report identified **LoanDefaultDetector version 14** as the
current production model.

Using the audit log, the system determined when the model was promoted
to production and who approved the deployment. The registry and MLflow
tracking data were then used to retrieve the dataset version used for
training and the evaluation metrics associated with the model.

This demonstrates how governance questions about model deployment,
approval history, training data, and performance can be answered
programmatically using the registry, experiment metadata, and lifecycle
audit records.

In [ ]:
#  AUTO-CHECK - Run this cell to verify your work

passed = []
failed = []

try:
    assert len(AUDIT_LOG) >= 2, 'Audit log must have at least 2 events'
    passed.append('Audit log has entries')
except Exception as e:
    failed.append(('Audit log has entries', str(e)))

try:
    assert all('timestamp' in e for e in AUDIT_LOG), 'Each event needs a timestamp'
    passed.append('Audit log has timestamps')
except Exception as e:
    failed.append(('Audit log has timestamps', str(e)))

try:
    assert (LAB_DIR / 'audit_log.json').exists(), 'audit_log.json not written to disk'
    passed.append('Audit log persisted')
except Exception as e:
    failed.append(('Audit log persisted', str(e)))

try:
    assert len(client.get_latest_versions(REGISTRY_NAME, stages=['Production'])) > 0, 'No model in Production stage'
    passed.append('Production model exists')
except Exception as e:
    failed.append(('Production model exists', str(e)))

try:
    assert len([e for e in AUDIT_LOG if 'ROLLBACK' in e.get('reason','')]) > 0, 'No rollback event in audit log'
    passed.append('Rollback executed')
except Exception as e:
    failed.append(('Rollback executed', str(e)))

print('\n' + '='*50)
print(f' Passed: {len(passed)}/{len(passed)+len(failed)}')
for p in passed: print(f'  {p}')
if failed:
    print('\nFailed:')
    for f,e in failed: print(f'   {f}: {e}')
else:
    print('\nAll checks passed!')

---
###  Advanced Path

Write the solution from scratch. Requirements listed below. No scaffolding provided.

---

###  Advanced Path - Task 4

Build a complete governance system:

**Requirements:**
1. `transition_stage()` with stage validation (only allow valid stage progressions: None→Staging, Staging→Production, Production→Archived)
2. `log_audit_event()` that writes to both in-memory list and a JSON file on disk
3. `rollback_production()` - archives current prod, promotes target, logs both events
4. `GovernanceReport` class with methods:
   - `current_production()` - returns name, version, stage, promoter, promotion time
   - `full_changelog(model_name)` - returns complete event history from audit log
   - `models_in_stage(stage)` - returns all models currently in a given stage
5. Simulate the full LoanGuard lifecycle: register v1 → staging → production → register bad v2 → production → rollback → archive v2
6. Print the final governance report answering all 5 questions from 4D

**Bonus:** Add a gate that blocks promotion to Production if the model's F1 score (from MLflow metrics) is below 0.65

## Explanation

This advanced implementation builds a complete model governance system
around the MLflow Model Registry.

The system enforces lifecycle rules for model promotion using a validated
stage transition function. Every lifecycle change is recorded through an
audit logging mechanism that stores events both in memory and in a
persistent JSON file.

A rollback mechanism restores a previous stable production model by
archiving the failing version and promoting the prior version back to
Production.

A `GovernanceReport` class provides programmatic access to governance
information such as the current production model, the complete lifecycle
changelog, and models currently deployed in a given stage.

Finally, the system simulates the full LoanGuard lifecycle including
deployment of a bad model, automatic rollback, and generation of a
governance report answering operational questions about the deployed
model.

A safety gate is also implemented to prevent promotion to Production if
the model's F1 score is below the required threshold.

In [ ]:
# ADVANCED - Task 4 Full Implementation
# -----------------------------------------------------------------

AUDIT_LOG = []
AUDIT_PATH = LAB_DIR / "audit_log.json"


# ---------------------------------------------------------------
# Audit logging
# ---------------------------------------------------------------

def log_audit_event(model_name, version, from_stage, to_stage, actor, reason):

    event = {
        "timestamp": datetime.now().isoformat(),
        "model_name": model_name,
        "version": version,
        "from_stage": from_stage,
        "to_stage": to_stage,
        "actor": actor,
        "reason": reason
    }

    AUDIT_LOG.append(event)

    if AUDIT_PATH.exists():
        with open(AUDIT_PATH, "r") as f:
            data = json.load(f)
    else:
        data = []

    data.append(event)

    with open(AUDIT_PATH, "w") as f:
        json.dump(data, f, indent=2)

    return event


# ---------------------------------------------------------------
# Lifecycle stage transitions with validation
# ---------------------------------------------------------------

VALID_TRANSITIONS = {
    "None": ["Staging"],
    "Staging": ["Production"],
    "Production": ["Archived"]
}


def transition_stage(model_name, version, stage, actor, reason):

    mv = client.get_model_version(model_name, version)
    current_stage = mv.current_stage or "None"

    if stage not in VALID_TRANSITIONS.get(current_stage, []):
        raise ValueError(f"Invalid transition {current_stage} → {stage}")

    client.transition_model_version_stage(model_name, version, stage)

    log_audit_event(model_name, version, current_stage, stage, actor, reason)

    print(f"{model_name} v{version} → {stage}")


# ---------------------------------------------------------------
# Bonus gate: block promotion if F1 < threshold
# ---------------------------------------------------------------

def check_metric_gate(run_id, threshold=0.65):

    run = client.get_run(run_id)
    f1 = run.data.metrics.get("f1", 0)

    if f1 < threshold:
        raise RuntimeError(
            f"Promotion blocked: F1={f1:.3f} below threshold {threshold}"
        )

    return f1


# ---------------------------------------------------------------
# Rollback function
# ---------------------------------------------------------------

def rollback_production(registry_name, rollback_version, actor, reason):

    prod_versions = client.get_latest_versions(registry_name, stages=["Production"])

    for v in prod_versions:

        client.transition_model_version_stage(
            registry_name,
            v.version,
            "Archived"
        )

        log_audit_event(
            registry_name,
            v.version,
            "Production",
            "Archived",
            actor,
            f"ROLLBACK: {reason}"
        )

    client.transition_model_version_stage(
        registry_name,
        rollback_version,
        "Production"
    )

    log_audit_event(
        registry_name,
        rollback_version,
        "Archived",
        "Production",
        actor,
        f"ROLLBACK TARGET: {reason}"
    )

    print(f"Rollback complete → version {rollback_version}")


# ---------------------------------------------------------------
# Governance report
# ---------------------------------------------------------------

class GovernanceReport:

    def current_production(self):

        prod = client.get_latest_versions(REGISTRY_NAME, stages=["Production"])

        if not prod:
            return None

        version = prod[0].version

        prod_event = next(
            (e for e in reversed(AUDIT_LOG)
             if e["version"] == version and e["to_stage"] == "Production"),
            None
        )

        return {
            "model": REGISTRY_NAME,
            "version": version,
            "stage": "Production",
            "promoter": prod_event["actor"],
            "promotion_time": prod_event["timestamp"]
        }

    def full_changelog(self, model_name):

        events = [e for e in AUDIT_LOG if e["model_name"] == model_name]

        lines = []
        for e in events:
            lines.append(
                f"{e['timestamp']} | v{e['version']} "
                f"{e['from_stage']} → {e['to_stage']} "
                f"| {e['actor']} | {e['reason']}"
            )

        return "\n".join(lines)

    def models_in_stage(self, stage):

        versions = client.get_latest_versions(REGISTRY_NAME, stages=[stage])
        return [(v.name, v.version) for v in versions]


# ---------------------------------------------------------------
# Simulate full lifecycle
# ---------------------------------------------------------------

print("Simulating LoanGuard lifecycle...")

best_run = client.get_run(BEST_RUN_ID)
f1 = best_run.data.metrics["f1"]

check_metric_gate(BEST_RUN_ID)

registered = mlflow.register_model(
    f"runs:/{BEST_RUN_ID}/model",
    REGISTRY_NAME
)

v1 = registered.version

transition_stage(
    REGISTRY_NAME,
    v1,
    "Staging",
    "ci-system",
    "Offline evaluation passed"
)

transition_stage(
    REGISTRY_NAME,
    v1,
    "Production",
    "ml-lead@loanGuard.ai",
    "Approved after staging validation"
)


# ---------------------------------------------------------------
# Deploy bad model (simulate failure)
# ---------------------------------------------------------------

weak_model = LogisticRegression(C=0.001, max_iter=100)

with mlflow.start_run() as bad_run:

    weak_model.fit(X_train, y_train)
    preds = weak_model.predict(X_test)

    mlflow.log_metric("f1", f1_score(y_test, preds))
    mlflow.sklearn.log_model(weak_model, "model")

    bad_run_id = bad_run.info.run_id


bad_reg = mlflow.register_model(
    f"runs:/{bad_run_id}/model",
    REGISTRY_NAME
)

v2 = bad_reg.version

transition_stage(
    REGISTRY_NAME,
    v2,
    "Staging",
    "ci-system",
    "Weak model accidentally approved"
)

transition_stage(
    REGISTRY_NAME,
    v2,
    "Production",
    "auto-deploy",
    "Deployment bug bypassed metric gate"
)

rollback_production(
    REGISTRY_NAME,
    rollback_version=v1,
    actor="ml-lead@loanGuard.ai",
    reason="Production F1 degraded"
)


# ---------------------------------------------------------------
# Final Governance Report
# ---------------------------------------------------------------

report = GovernanceReport()

print("\nFinal Governance Report")
print("=" * 60)

prod = report.current_production()

print(f"Current production model : {prod['model']} v{prod['version']}")
print(f"Promoted by              : {prod['promoter']}")
print(f"Promotion time           : {prod['promotion_time']}")

print("\nModels currently in Production:")
print(report.models_in_stage("Production"))

print("\nComplete changelog:")
print(report.full_changelog(REGISTRY_NAME))

## Observation

The advanced governance system successfully implemented lifecycle
management, audit logging, rollback functionality, and automated
governance reporting for models registered in the MLflow Model Registry.

During the lifecycle simulation, model **version 22** was registered,
validated in the **Staging** stage, and approved for **Production**.
A second model (**version 23**) was later deployed due to an automated
pipeline bug, which resulted in degraded performance.

The rollback mechanism correctly detected the issue and restored the
previous stable model (**version 22**) to the Production stage while
archiving the faulty deployment. All lifecycle events—including staging
approval, production deployment, failure, and rollback—were captured in
the audit log.

The final governance report programmatically identified the active
production model, the approver responsible for the deployment, the
promotion timestamp, and the full lifecycle changelog. This demonstrates
a complete model governance pipeline capable of ensuring traceability,
deployment safety, and operational recovery in a production ML system.

---
#  Lab Complete - Reflection
---

##  What You Built

| Task | What you implemented | Real-world equivalent |
|------|----------------------|----------------------|
| **Task 1** | DVC pointer files, drift detection, schema validator | DVC + Great Expectations |
| **Task 2** | MLflow experiment runs, multi-model comparison, artifact retrieval | MLflow Tracking Server |
| **Task 3** | Versioned model artifacts, semantic version logic, model registration | MLflow Registry + release process |
| **Task 4** | Stage transitions, audit log, rollback, governance queries | MLflow Registry Lifecycle + compliance |

---

## The Full Stack - How It Connects

```
loans_v1.csv  →  validate_schema()  →  [PASSES]  →  Training Run
                                                         │
                                              mlflow.log_params()
                                              mlflow.log_metrics()
                                              mlflow.log_model()
                                                         │
                                              register_model("LoanDefaultDetector")
                                                         │
                                         None → Staging → Production
                                                    (with audit log)
                                                         │
                             If it fails →  rollback_production()  →  Archived
```

---

##  Reflection Questions

Answer these in the cell below before the session ends:

1. Which task felt most like something you'd use immediately in your current project?
2. Where in LoanGuard's original problem could a schema validator have saved the most time?
3. If you had to pick ONE layer of the versioning stack to implement tomorrow - which would it be and why?

In [ ]:
#  Your Reflection Answers
# ----------------------------------------------------

reflection = {
    "Which task felt most like something you'd use immediately in your current project?": """
    Task 2 (MLflow experiment tracking) felt the most immediately useful.
    Being able to log parameters, metrics, and models for every training
    run makes experimentation reproducible and removes guesswork when
    comparing models. It also makes collaboration easier because other
    engineers can retrieve the exact model and configuration that
    produced a specific result.
    """,

    "Where in LoanGuard's original problem could a schema validator have saved the most time?": """
    A schema validator would have helped immediately when the upstream
    dataset changed between versions (for example when new columns were
    added or null values appeared in the credit_score field). Instead of
    silently training on corrupted or incomplete data, the pipeline could
    have failed early and alerted the team before the degraded model was
    deployed.
    """,

    "If you had to pick ONE layer of the versioning stack to implement tomorrow - which would it be and why?": """
    I would implement experiment tracking (MLflow) first. Tracking runs,
    parameters, metrics, and model artifacts creates a reliable history
    of experiments and makes it easy to identify the best performing
    models. Once that foundation exists, adding data versioning and
    lifecycle governance becomes much easier.
    """
}

for q, a in reflection.items():
    print(f"Q: {q}")
    print(f"A: {a.strip()}")
    print()

# Lab Reflection

In this lab we simulated a real-world MLOps workflow used in production machine learning systems.

Key components implemented:

• Data versioning using DVC-style pointer files  
• Dataset drift detection across dataset versions  
• Schema validation to enforce data contracts  
• Experiment tracking using MLflow  
• Comparison of multiple models and evaluation metrics  
• Model artifact versioning using semantic versioning

This workflow closely mirrors real-world MLOps stacks used in industry, such as:

DVC + Great Expectations + MLflow + Model Registry

By combining data versioning, experiment tracking, and model versioning, we ensure that
machine learning systems remain reproducible, traceable, and production-ready.

**Note:Used AI for writing explanations and observation cells correctly**